In [1]:
import os
import re
import math
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm
from sklearn.model_selection import train_test_split

# 재현성을 위한 시드 고정
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 디바이스: {device}")


PyTorch 버전: 2.7.1+cu118
사용 디바이스: cuda


In [2]:
import os
import urllib.request
import zipfile

DATA_DIR = '/home/jovyan/work/transformer_chatbot/data/'
os.makedirs(DATA_DIR, exist_ok=True)

# ── 1. 한국어 챗봇 데이터 다운로드 ────────────────────────────────────────
# 출처: https://github.com/songys/Chatbot_data
# Q(질문), A(답변), label(감정) 3컬럼, 11,823개
print("한국어 챗봇 데이터 다운로드 중...")
urllib.request.urlretrieve(
    "https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv",
    DATA_DIR + 'ChatbotData.csv'
)
print("완료!")

# ── 2. 코넬 영화 대화 데이터 다운로드 ─────────────────────────────────────
# 출처: http://www.cs.cornell.edu/~cristian/Cornell_Movie-Dialogs_Corpus.html
# 영화 대사 기반 대화 데이터, 최대 50,000쌍 사용
zip_path = DATA_DIR + 'cornell_movie_dialogs.zip'

if not os.path.exists(zip_path):
    print("코넬 데이터 다운로드 중...")
    urllib.request.urlretrieve(
        'http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip',
        zip_path
    )
    print("완료!")

# ZIP 압축 해제
print("압축 해제 중...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)
print("완료!")

# 파일 경로 확인
cornell_dir         = DATA_DIR + 'cornell movie-dialogs corpus/'
path_to_lines       = cornell_dir + 'movie_lines.txt'
path_to_convs       = cornell_dir + 'movie_conversations.txt'

print(f"\n한국어 데이터: {DATA_DIR + 'ChatbotData.csv'}")
print(f"코넬 라인 파일: {path_to_lines}")
print(f"코넬 대화 파일: {path_to_convs}")

# 파일 존재 여부 확인
print(f"\n파일 존재 여부:")
print(f"  ChatbotData.csv: {os.path.exists(DATA_DIR + 'ChatbotData.csv')}")
print(f"  movie_lines.txt: {os.path.exists(path_to_lines)}")
print(f"  movie_conversations.txt: {os.path.exists(path_to_convs)}")

한국어 챗봇 데이터 다운로드 중...
완료!
코넬 데이터 다운로드 중...
완료!
압축 해제 중...
완료!

한국어 데이터: /home/jovyan/work/transformer_chatbot/data/ChatbotData.csv
코넬 라인 파일: /home/jovyan/work/transformer_chatbot/data/cornell movie-dialogs corpus/movie_lines.txt
코넬 대화 파일: /home/jovyan/work/transformer_chatbot/data/cornell movie-dialogs corpus/movie_conversations.txt

파일 존재 여부:
  ChatbotData.csv: True
  movie_lines.txt: True
  movie_conversations.txt: True


In [3]:
# 사용할 최대 샘플 수
# 코넬 데이터는 22만 쌍이지만 학습 시간 고려해서 50,000개만 사용
MAX_SAMPLES = 50000

# ── 전처리 함수 ────────────────────────────────────────────────────────────
# 영어 전처리: 소문자 변환, 구두점 앞뒤 공백 추가, 특수문자 제거
def preprocess_en(sentence):
    sentence = sentence.lower().strip()
    # 구두점 앞뒤에 공백 추가 (예: "hello." → "hello .")
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    # 영문, 숫자, 기본 구두점만 유지
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)
    return sentence.strip()

# ── 코넬 데이터 로드 ───────────────────────────────────────────────────────
def load_cornell_data(path_to_lines, path_to_convs, max_samples=50000):
    # 1) line_id → 대사 텍스트 딕셔너리 생성
    id2line = {}
    with open(path_to_lines, 'r', errors='ignore') as f:
        for line in f:
            parts = line.strip().split(" +++$+++ ")
            if len(parts) >= 5:
                id2line[parts[0]] = parts[4]

    # 2) 대화에서 연속된 두 문장을 (질문, 답변) 쌍으로 추출
    pairs = []
    with open(path_to_convs, 'r', errors='ignore') as f:
        for line in f:
            parts = line.strip().split(" +++$+++ ")
            if len(parts) < 4:
                continue
            # 대화 ID 목록 파싱: "['L1', 'L2', 'L3']" → ['L1', 'L2', 'L3']
            conv_list = [c.strip("' ") for c in parts[3][1:-1].split(", ")]
            for i in range(len(conv_list) - 1):
                q = preprocess_en(id2line.get(conv_list[i],   ""))
                a = preprocess_en(id2line.get(conv_list[i+1], ""))
                # 빈 문장 제거
                if q and a:
                    pairs.append((q, a))
                if len(pairs) >= max_samples:
                    return pairs
    return pairs

print("코넬 데이터 로드 중...")
cornell_pairs = load_cornell_data(path_to_lines, path_to_convs, MAX_SAMPLES)
print(f"코넬 데이터 샘플 수: {len(cornell_pairs)}")
print(f"샘플 예시:")
for q, a in cornell_pairs[:3]:
    print(f"  Q: {q}")
    print(f"  A: {a}")
    print()

코넬 데이터 로드 중...
코넬 데이터 샘플 수: 50000
샘플 예시:
  Q: can we make this quick ? roxanne korrine and andrew barrett are having an incredibly horrendous public break up on the quad . again .
  A: well , i thought we d start with pronunciation , if that s okay with you .

  Q: well , i thought we d start with pronunciation , if that s okay with you .
  A: not the hacking and gagging and spitting part . please .

  Q: not the hacking and gagging and spitting part . please .
  A: okay . . . then how bout we try out some french cuisine . saturday ? night ?



In [4]:
# 특수 토큰 ID 설정
PAD_ID = 0  # 패딩 토큰
BOS_ID = 1  # 문장 시작 토큰
EOS_ID = 2  # 문장 종료 토큰
UNK_ID = 3  # 미등록 토큰

# ── SentencePiece 학습용 코퍼스 생성 ──────────────────────────────────────
# 코넬 데이터의 Q, A를 모두 합쳐서 학습
# → 질문/답변에 나오는 모든 단어를 vocab에 포함시키기 위함
corpus_path = DATA_DIR + 'corpus.txt'
with open(corpus_path, 'w', encoding='utf-8') as f:
    for q, a in cornell_pairs:
        f.write(q.strip() + '\n')
        f.write(a.strip() + '\n')

print(f"코퍼스 문장 수: {len(cornell_pairs) * 2:,}")

# ── SentencePiece BPE 모델 학습 ────────────────────────────────────────────
# character_coverage=1.0: 영어는 알파벳이 적으므로 1.0 설정 (한국어는 0.9995)
# model_type='bpe': 자주 등장하는 문자 쌍을 합쳐서 서브워드 생성
spm.SentencePieceTrainer.Train(
    input=corpus_path,
    model_prefix=DATA_DIR + 'spm_cornell',
    vocab_size=8000,
    character_coverage=1.0,
    model_type='bpe',
    pad_id=PAD_ID,
    bos_id=BOS_ID,
    eos_id=EOS_ID,
    unk_id=UNK_ID,
    pad_piece='[PAD]',
    bos_piece='[BOS]',
    eos_piece='[EOS]',
    unk_piece='[UNK]',
)

# 학습된 모델 로드
sp = spm.SentencePieceProcessor()
sp.Load(DATA_DIR + 'spm_cornell.model')

# 동작 확인
test = "can we make this quick ?"
print(f"\n원문: {test}")
print(f"토크나이징: {sp.encode_as_pieces(test)}")
print(f"ID 변환: {sp.encode_as_ids(test)}")
print(f"vocab 크기: {sp.get_piece_size()}")

코퍼스 문장 수: 100,000

원문: can we make this quick ?
토크나이징: ['▁can', '▁we', '▁make', '▁this', '▁quick', '▁?']
ID 변환: [115, 52, 321, 99, 1610, 23]
vocab 크기: 8000


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /home/jovyan/work/transformer_chatbot/data/corpus.txt
  input_format: 
  model_prefix: /home/jovyan/work/transformer_chatbot/data/spm_cornell
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: [UNK]
  bos_piece: [BOS]
  eos_piece: [EOS

In [5]:
# 하이퍼파라미터 설정
VOCAB_SIZE = 8000  # 토크나이저 vocab 크기
MAX_LEN    = 40    # 최대 시퀀스 길이
# 영어 데이터는 한국어보다 토큰이 많으므로 40으로 설정
BATCH_SIZE = 64    # 한 번에 학습할 데이터 수

class CornellDataset(Dataset):
    def __init__(self, pairs, sp, max_len):
        self.data = []
        for q, a in pairs:
            # 인코더 입력: [BOS] + Q 토큰 + [EOS] + 패딩
            enc_ids = [BOS_ID] + sp.encode_as_ids(q) + [EOS_ID]
            enc_ids = enc_ids[:max_len]
            enc_ids += [PAD_ID] * (max_len - len(enc_ids))

            # 디코더 입력: [BOS] + A 토큰 (정답을 한 칸 오른쪽으로 밀어서 입력)
            dec_ids = [BOS_ID] + sp.encode_as_ids(a)
            dec_ids = dec_ids[:max_len]
            dec_ids += [PAD_ID] * (max_len - len(dec_ids))

            # 정답 레이블: A 토큰 + [EOS] (모델이 예측해야 할 값)
            # 예: dec_input=[BOS, I, am], label=[I, am, EOS]
            label_ids = sp.encode_as_ids(a) + [EOS_ID]
            label_ids = label_ids[:max_len]
            label_ids += [PAD_ID] * (max_len - len(label_ids))

            self.data.append((
                torch.tensor(enc_ids,   dtype=torch.long),
                torch.tensor(dec_ids,   dtype=torch.long),
                torch.tensor(label_ids, dtype=torch.long),
            ))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# 학습/검증 데이터 분리 (9:1)
train_pairs, val_pairs = train_test_split(cornell_pairs, test_size=0.1, random_state=42)

train_dataset = CornellDataset(train_pairs, sp, MAX_LEN)
val_dataset   = CornellDataset(val_pairs,   sp, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

print(f"학습 데이터: {len(train_dataset):,}개")
print(f"검증 데이터: {len(val_dataset):,}개")

# 샘플 확인
enc, dec, label = train_dataset[0]
print(f"\n인코더 디코딩: {sp.decode_ids(enc.tolist())}")
print(f"디코더 디코딩: {sp.decode_ids(dec.tolist())}")
print(f"레이블 디코딩: {sp.decode_ids(label.tolist())}")

# PAD 비율 확인
sample_dec = torch.stack([train_dataset[i][1] for i in range(200)])
pad_ratio  = (sample_dec == PAD_ID).float().mean().item()
print(f"\nPAD 비율: {pad_ratio * 100:.1f}%")

학습 데이터: 45,000개
검증 데이터: 5,000개

인코더 디코딩: i ll help you in .
디코더 디코딩: kay em , you ve saved our lives , you know that don t you ?
레이블 디코딩: kay em , you ve saved our lives , you know that don t you ?

PAD 비율: 63.4%


In [1]:
# ── Scaled Dot-Product Attention ──────────────────────────────────────────
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    query, key, value: (batch, heads, seq_len, depth)
    1) Q·Kᵀ 로 유사도 점수 계산
    2) √dₖ 로 스케일링 (값이 너무 커지면 softmax 기울기 소실 방지)
    3) 마스크 적용 (PAD 또는 미래 토큰 차단)
    4) softmax → 어텐션 가중치
    5) 가중치 · V → 최종 출력
    """
    d_k    = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, value), weights


# ── Multi-Head Attention ───────────────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    """
    num_heads개의 어텐션을 병렬로 수행
    각 헤드가 서로 다른 관점(위치, 의미 등)을 학습
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.depth     = d_model // num_heads  # 각 헤드의 차원

        self.wq = nn.Linear(d_model, d_model)  # Query 변환
        self.wk = nn.Linear(d_model, d_model)  # Key 변환
        self.wv = nn.Linear(d_model, d_model)  # Value 변환
        self.wo = nn.Linear(d_model, d_model)  # 헤드 합친 후 최종 변환

    def split_heads(self, x, batch):
        # (batch, seq_len, d_model) → (batch, heads, seq_len, depth)
        x = x.view(batch, -1, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)

    def forward(self, q, k, v, mask=None):
        batch = q.size(0)
        q = self.split_heads(self.wq(q), batch)
        k = self.split_heads(self.wk(k), batch)
        v = self.split_heads(self.wv(v), batch)

        out, _ = scaled_dot_product_attention(q, k, v, mask)
        # (batch, heads, seq_len, depth) → (batch, seq_len, d_model)
        out = out.permute(0, 2, 1, 3).contiguous().view(batch, -1, self.num_heads * self.depth)
        return self.wo(out)


# ── Positional Encoding ────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    """
    트랜스포머는 순서 정보가 없으므로 위치 정보를 직접 더해줌
    sin/cos 함수로 각 위치마다 고유한 패턴 생성
    position=vocab_size로 설정 (참고 코드 방식)
    """
    def __init__(self, position, d_model, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(position, d_model)
        pos = torch.arange(0, position).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)  # 짝수 인덱스 → sin
        pe[:, 1::2] = torch.cos(pos * div)  # 홀수 인덱스 → cos
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, position, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ── Encoder Layer ──────────────────────────────────────────────────────────
class EncoderLayer(nn.Module):
    """
    인코더 한 층 = Self-Attention + FFN
    각 서브레이어마다 잔차 연결 + LayerNorm 적용
    → 기울기 소실 방지 및 학습 안정화
    """
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha      = MultiHeadAttention(d_model, num_heads)
        self.ffn      = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model)
        )
        self.norm1    = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2    = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout1(self.mha(x, x, x, mask)))  # Self-Attention + 잔차
        x = self.norm2(x + self.dropout2(self.ffn(x)))               # FFN + 잔차
        return x


# ── Decoder Layer ──────────────────────────────────────────────────────────
class DecoderLayer(nn.Module):
    """
    디코더 한 층 = Masked Self-Attention + Cross-Attention + FFN
    - Masked Self-Attention: 미래 토큰 차단
    - Cross-Attention: 인코더 출력 참조 (Q=디코더, K/V=인코더)
    """
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha1     = MultiHeadAttention(d_model, num_heads)  # Masked Self-Attention
        self.mha2     = MultiHeadAttention(d_model, num_heads)  # Cross-Attention
        self.ffn      = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model)
        )
        self.norm1    = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2    = nn.LayerNorm(d_model, eps=1e-6)
        self.norm3    = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, look_ahead_mask=None, padding_mask=None):
        x = self.norm1(x + self.dropout1(self.mha1(x, x, x, look_ahead_mask)))        # Masked Self-Attention
        x = self.norm2(x + self.dropout2(self.mha2(x, enc_out, enc_out, padding_mask))) # Cross-Attention
        x = self.norm3(x + self.dropout3(self.ffn(x)))                                  # FFN
        return x


# ── Encoder ────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.d_model      = d_model
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoding = PositionalEncoding(vocab_size, d_model, dropout)
        self.layers       = nn.ModuleList([EncoderLayer(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.dropout      = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # √d_model 스케일링: 임베딩 값이 작아서 위치 인코딩에 묻히는 것 방지
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, mask)
        return x


# ── Decoder ────────────────────────────────────────────────────────────────
class Decoder(nn.Module):
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.d_model      = d_model
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoding = PositionalEncoding(vocab_size, d_model, dropout)
        self.layers       = nn.ModuleList([DecoderLayer(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.dropout      = nn.Dropout(dropout)

    def forward(self, x, enc_out, look_ahead_mask=None, padding_mask=None):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, enc_out, look_ahead_mask, padding_mask)
        return x


# ── Transformer ────────────────────────────────────────────────────────────
class Transformer(nn.Module):
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.encoder     = Encoder(vocab_size, num_layers, d_model, num_heads, ff_dim, dropout)
        self.decoder     = Decoder(vocab_size, num_layers, d_model, num_heads, ff_dim, dropout)
        self.final_layer = nn.Linear(d_model, vocab_size)  # vocab 크기로 최종 변환

    def forward(self, src, tgt, src_mask=None, look_ahead_mask=None):
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt, enc_out, look_ahead_mask, src_mask)
        return self.final_layer(dec_out)  # (batch, tgt_len, vocab_size)


# ── 마스크 함수 ────────────────────────────────────────────────────────────
def create_padding_mask(x):
    """PAD 위치를 0으로 마스킹 → 어텐션 계산에서 PAD 무시"""
    mask = (x != PAD_ID).float()
    return mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)

def create_look_ahead_mask(x):
    """
    미래 토큰 차단 마스크
    예: 3번째 토큰 예측 시 1, 2번째만 참조 가능
    """
    seq_len     = x.size(1)
    look_ahead  = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
    look_ahead  = look_ahead.unsqueeze(0).unsqueeze(1)  # (1, 1, seq_len, seq_len)
    padding     = create_padding_mask(x)                 # (batch, 1, 1, seq_len)
    return torch.min(look_ahead, padding)                # 둘 다 1일 때만 통과


# ── 하이퍼파라미터 및 모델 생성 ───────────────────────────────────────────
NUM_LAYERS = 2    # 인코더/디코더 레이어 수
D_MODEL    = 256  # 임베딩 차원
NUM_HEADS  = 8    # 어텐션 헤드 수 (D_MODEL % NUM_HEADS == 0 이어야 함)
FF_DIM     = 512  # FFN 중간 차원 (보통 D_MODEL * 2~4)
DROPOUT    = 0.1  # 드롭아웃 비율

model = Transformer(
    vocab_size = VOCAB_SIZE,
    num_layers = NUM_LAYERS,
    d_model    = D_MODEL,
    num_heads  = NUM_HEADS,
    ff_dim     = FF_DIM,
    dropout    = DROPOUT
).to(device)

print(f"파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# 동작 확인
src = torch.randint(0, VOCAB_SIZE, (2, MAX_LEN)).to(device)
tgt = torch.randint(0, VOCAB_SIZE, (2, MAX_LEN)).to(device)
out = model(src, tgt)
print(f"출력 shape: {out.shape}")  # (2, MAX_LEN, VOCAB_SIZE)

NameError: name 'nn' is not defined

In [8]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    query, key, value: (batch, heads, seq_len, depth)
    1) Q·Kᵀ 로 각 토큰 간 유사도 점수 계산
    2) √dₖ 로 나눠서 스케일링 (값이 커지면 softmax 기울기 소실)
    3) 마스크 적용 (PAD 또는 미래 토큰 차단)
    4) softmax → 어텐션 가중치 (합이 1인 확률 분포)
    5) 가중치 · V → 최종 출력
    """
    d_k = query.size(-1)
    # (batch, heads, seq_len_q, seq_len_k)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        # 마스크가 0인 위치에 매우 작은 값 → softmax 후 0에 가까워짐
        scores = scores.masked_fill(mask == 0, -1e9)

    weights = F.softmax(scores, dim=-1)
    output  = torch.matmul(weights, value)
    return output, weights

# 동작 확인
q = torch.randn(2, 8, 10, 32)  # (batch=2, heads=8, seq_len=10, depth=32)
k = torch.randn(2, 8, 10, 32)
v = torch.randn(2, 8, 10, 32)
out, w = scaled_dot_product_attention(q, k, v)
print(f"Attention 출력 shape: {out.shape}")     # (2, 8, 10, 32)
print(f"Attention 가중치 shape: {w.shape}")     # (2, 8, 10, 10)
print(f"가중치 합 (1이어야 함): {w[0,0,0].sum():.4f}")

Attention 출력 shape: torch.Size([2, 8, 10, 32])
Attention 가중치 shape: torch.Size([2, 8, 10, 10])
가중치 합 (1이어야 함): 1.0000


In [9]:
class MultiHeadAttention(nn.Module):
    """
    하나의 어텐션 대신 num_heads개로 나눠서 병렬 계산
    각 헤드가 서로 다른 관점(위치, 의미 등)을 학습함
    예: 8헤드면 d_model=256을 32씩 8개로 나눠서 각각 어텐션 계산
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model은 num_heads로 나누어 떨어져야 함"
        self.num_heads = num_heads
        self.depth     = d_model // num_heads  # 각 헤드의 차원

        self.wq = nn.Linear(d_model, d_model)  # Query 변환 행렬
        self.wk = nn.Linear(d_model, d_model)  # Key 변환 행렬
        self.wv = nn.Linear(d_model, d_model)  # Value 변환 행렬
        self.wo = nn.Linear(d_model, d_model)  # 헤드 합친 후 최종 변환

    def split_heads(self, x, batch_size):
        """
        (batch, seq_len, d_model) → (batch, heads, seq_len, depth)
        헤드별로 나눠서 병렬 어텐션 계산 가능하게 함
        """
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)

    def forward(self, q, k, v, mask=None):
        batch = q.size(0)

        # Q, K, V 각각 Linear 변환 후 헤드 분할
        q = self.split_heads(self.wq(q), batch)
        k = self.split_heads(self.wk(k), batch)
        v = self.split_heads(self.wv(v), batch)

        # 스케일드 닷 프로덕트 어텐션
        out, _ = scaled_dot_product_attention(q, k, v, mask)

        # (batch, heads, seq_len, depth) → (batch, seq_len, d_model)
        out = out.permute(0, 2, 1, 3).contiguous()
        out = out.view(batch, -1, self.num_heads * self.depth)

        return self.wo(out)

# 동작 확인
mha = MultiHeadAttention(d_model=256, num_heads=8)
x   = torch.randn(2, 10, 256)  # (batch=2, seq_len=10, d_model=256)
out = mha(x, x, x)
print(f"MultiHeadAttention 출력 shape: {out.shape}")  # (2, 10, 256)

MultiHeadAttention 출력 shape: torch.Size([2, 10, 256])


In [10]:
class PositionalEncoding(nn.Module):
    """
    트랜스포머는 RNN과 달리 순서 정보가 없음
    → 각 위치마다 고유한 sin/cos 패턴을 임베딩에 더해서 위치 정보 제공
    position=vocab_size로 설정하는 이유:
    최대한 많은 위치를 커버하기 위함 (실제 사용은 seq_len만큼만)
    """
    def __init__(self, position, d_model, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # (position, d_model) 크기의 PE 행렬 생성
        pe  = torch.zeros(position, d_model)
        pos = torch.arange(0, position).unsqueeze(1).float()  # (position, 1)
        # 각 차원마다 다른 주기의 sin/cos 적용
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)  # 짝수 인덱스 → sin
        pe[:, 1::2] = torch.cos(pos * div)  # 홀수 인덱스 → cos

        # (1, position, d_model) → 배치 브로드캐스팅용
        pe = pe.unsqueeze(0)
        # register_buffer: 학습 파라미터는 아니지만 모델과 함께 저장/이동
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        # 임베딩에 위치 인코딩 더하기 (seq_len만큼만 슬라이싱)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# 동작 확인
pe  = PositionalEncoding(position=VOCAB_SIZE, d_model=256)
x   = torch.zeros(2, 10, 256)  # (batch=2, seq_len=10, d_model=256)
out = pe(x)
print(f"PositionalEncoding 출력 shape: {out.shape}")  # (2, 10, 256)
# 위치마다 다른 값이 들어갔는지 확인
print(f"위치 0과 위치 1이 다른지: {not torch.equal(out[0,0], out[0,1])}")

PositionalEncoding 출력 shape: torch.Size([2, 10, 256])
위치 0과 위치 1이 다른지: True


In [11]:
class EncoderLayer(nn.Module):
    """
    인코더 한 층 = Self-Attention + FFN
    - Self-Attention: 입력 문장 내 모든 토큰 간 관계 학습
    - FFN: 각 토큰의 표현을 비선형 변환으로 풍부하게 만듦
    - 잔차 연결(x + sublayer(x)): 기울기 소실 방지
    - LayerNorm: 학습 안정화
    """
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha      = MultiHeadAttention(d_model, num_heads)
        self.ffn      = nn.Sequential(
            nn.Linear(d_model, ff_dim),  # d_model → ff_dim으로 확장
            nn.ReLU(),                   # 비선형 활성화
            nn.Linear(ff_dim, d_model)   # ff_dim → d_model로 복원
        )
        self.norm1    = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2    = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 1) Self-Attention + 잔차 연결 + LayerNorm
        attn_out = self.mha(x, x, x, mask)        # Q=K=V=x (Self-Attention)
        attn_out = self.dropout1(attn_out)
        out1     = self.norm1(x + attn_out)        # 잔차 연결

        # 2) FFN + 잔차 연결 + LayerNorm
        ffn_out  = self.ffn(out1)
        ffn_out  = self.dropout2(ffn_out)
        out2     = self.norm2(out1 + ffn_out)      # 잔차 연결

        return out2

# 동작 확인
enc_layer = EncoderLayer(d_model=256, num_heads=8, ff_dim=512)
x         = torch.randn(2, 10, 256)  # (batch=2, seq_len=10, d_model=256)
out       = enc_layer(x)
print(f"EncoderLayer 출력 shape: {out.shape}")  # (2, 10, 256)

EncoderLayer 출력 shape: torch.Size([2, 10, 256])


In [12]:
class DecoderLayer(nn.Module):
    """
    디코더 한 층 = Masked Self-Attention + Cross-Attention + FFN
    - Masked Self-Attention: 미래 토큰을 보지 못하도록 마스킹
      예: "I am a" 다음 토큰 예측 시 "student" 는 볼 수 없음
    - Cross-Attention: 인코더 출력을 K, V로 사용
      → 입력 문장(Q)과 출력 문장(K,V) 간의 관계 학습
    - FFN: 각 토큰 표현을 비선형 변환
    """
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        # 1) Masked Self-Attention (디코더 내부)
        self.mha1     = MultiHeadAttention(d_model, num_heads)
        # 2) Cross-Attention (인코더-디코더)
        self.mha2     = MultiHeadAttention(d_model, num_heads)
        # 3) FFN
        self.ffn      = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )
        self.norm1    = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2    = nn.LayerNorm(d_model, eps=1e-6)
        self.norm3    = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, look_ahead_mask=None, padding_mask=None):
        # 1) Masked Self-Attention (미래 토큰 차단)
        # Q=K=V=x, look_ahead_mask로 미래 토큰 차단
        attn1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1)
        out1  = self.norm1(x + attn1)

        # 2) Cross-Attention (Q=디코더, K/V=인코더 출력)
        # 디코더가 인코더의 입력 문장을 참조하는 부분
        attn2 = self.mha2(out1, enc_out, enc_out, padding_mask)
        attn2 = self.dropout2(attn2)
        out2  = self.norm2(out1 + attn2)

        # 3) FFN
        ffn_out = self.ffn(out2)
        ffn_out = self.dropout3(ffn_out)
        out3    = self.norm3(out2 + ffn_out)

        return out3

# 동작 확인
dec_layer = DecoderLayer(d_model=256, num_heads=8, ff_dim=512)
x         = torch.randn(2, 10, 256)   # 디코더 입력 (batch=2, seq_len=10, d_model=256)
enc_out   = torch.randn(2, 10, 256)   # 인코더 출력
out       = dec_layer(x, enc_out)
print(f"DecoderLayer 출력 shape: {out.shape}")  # (2, 10, 256)

DecoderLayer 출력 shape: torch.Size([2, 10, 256])


In [13]:
class Encoder(nn.Module):
    """
    EncoderLayer를 num_layers개 쌓은 전체 인코더
    입력 문장을 받아서 의미있는 벡터 표현으로 변환
    1) 임베딩: 토큰 ID → 벡터
    2) √d_model 스케일링: 임베딩 값이 작아서 위치 인코딩에 묻히는 걸 방지
    3) 위치 인코딩: 순서 정보 추가
    4) EncoderLayer × num_layers: 점점 깊은 표현 학습
    """
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.d_model      = d_model
        # padding_idx: PAD 토큰은 항상 0벡터로 임베딩 (학습 안 함)
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoding = PositionalEncoding(vocab_size, d_model, dropout)
        self.layers       = nn.ModuleList([
            EncoderLayer(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.dropout      = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 1) 임베딩 + √d_model 스케일링
        x = self.embedding(x) * math.sqrt(self.d_model)
        # 2) 위치 인코딩
        x = self.pos_encoding(x)
        x = self.dropout(x)
        # 3) EncoderLayer 순서대로 통과
        for layer in self.layers:
            x = layer(x, mask)
        return x  # (batch, seq_len, d_model)

# 동작 확인
encoder = Encoder(
    vocab_size = VOCAB_SIZE,
    num_layers = 2,
    d_model    = 256,
    num_heads  = 8,
    ff_dim     = 512
)
x   = torch.randint(0, VOCAB_SIZE, (2, MAX_LEN))  # (batch=2, seq_len=40)
out = encoder(x)
print(f"Encoder 출력 shape: {out.shape}")  # (2, 40, 256)

Encoder 출력 shape: torch.Size([2, 40, 256])


In [14]:
class Decoder(nn.Module):
    """
    DecoderLayer를 num_layers개 쌓은 전체 디코더
    인코더 출력과 이전까지 생성한 토큰을 받아서 다음 토큰 예측
    1) 임베딩 + √d_model 스케일링
    2) 위치 인코딩
    3) DecoderLayer × num_layers
       - 각 층마다 인코더 출력(enc_out)을 Cross-Attention으로 참조
    """
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.d_model      = d_model
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoding = PositionalEncoding(vocab_size, d_model, dropout)
        self.layers       = nn.ModuleList([
            DecoderLayer(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, look_ahead_mask=None, padding_mask=None):
        # 1) 임베딩 + √d_model 스케일링
        x = self.embedding(x) * math.sqrt(self.d_model)
        # 2) 위치 인코딩
        x = self.pos_encoding(x)
        x = self.dropout(x)
        # 3) DecoderLayer 순서대로 통과
        # 매 층마다 인코더 출력을 Cross-Attention으로 참조
        for layer in self.layers:
            x = layer(x, enc_out, look_ahead_mask, padding_mask)
        return x  # (batch, tgt_seq_len, d_model)

# 동작 확인
decoder = Decoder(
    vocab_size = VOCAB_SIZE,
    num_layers = 2,
    d_model    = 256,
    num_heads  = 8,
    ff_dim     = 512
)
x       = torch.randint(0, VOCAB_SIZE, (2, MAX_LEN))  # 디코더 입력
enc_out = torch.randn(2, MAX_LEN, 256)                # 인코더 출력
out     = decoder(x, enc_out)
print(f"Decoder 출력 shape: {out.shape}")  # (2, 40, 256)

Decoder 출력 shape: torch.Size([2, 40, 256])


In [15]:
class Transformer(nn.Module):
    """
    인코더 + 디코더 + 최종 출력층으로 구성된 전체 트랜스포머 모델
    - 인코더: 입력 문장 → 의미 벡터
    - 디코더: 의미 벡터 + 이전 출력 → 다음 토큰 예측
    - final_layer: d_model → vocab_size (각 토큰의 확률 분포)
    """
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.encoder     = Encoder(vocab_size, num_layers, d_model, num_heads, ff_dim, dropout)
        self.decoder     = Decoder(vocab_size, num_layers, d_model, num_heads, ff_dim, dropout)
        # 최종 출력: (batch, seq_len, d_model) → (batch, seq_len, vocab_size)
        self.final_layer = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, src_mask=None, look_ahead_mask=None):
        # 1) 인코더: 입력 문장 인코딩
        enc_out = self.encoder(src, src_mask)
        # 2) 디코더: 인코더 출력 참조하며 출력 생성
        # src_mask를 dec_padding_mask로도 사용
        # (인코더 출력에서 PAD 위치를 Cross-Attention에서 무시하기 위함)
        dec_out = self.decoder(tgt, enc_out, look_ahead_mask, src_mask)
        # 3) 최종 Linear → vocab_size 크기의 logit 반환
        return self.final_layer(dec_out)  # (batch, tgt_len, vocab_size)

# ── 마스크 함수 ─────────────────────────────────────────────────────────────
def create_padding_mask(x):
    """
    PAD 토큰 위치를 0으로 마스킹
    어텐션 계산 시 PAD 위치는 -1e9로 채워져 softmax 후 0에 가까워짐
    """
    mask = (x != PAD_ID).float()
    return mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)

def create_look_ahead_mask(x):
    """
    디코더에서 미래 토큰을 보지 못하도록 상삼각 마스킹
    하삼각행렬: 자기 자신 포함 이전 토큰만 1
    예: 3번째 토큰 예측 시 1, 2번째만 볼 수 있음
    패딩 마스크와 합쳐서 PAD도 동시에 차단
    """
    seq_len     = x.size(1)
    look_ahead  = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
    look_ahead  = look_ahead.unsqueeze(0).unsqueeze(1)  # (1, 1, seq_len, seq_len)
    padding     = create_padding_mask(x)                # (batch, 1, 1, seq_len)
    # 둘 중 하나라도 0이면 차단 (min = 둘 다 1일 때만 1)
    return torch.min(look_ahead, padding)

# ── 하이퍼파라미터 설정 ─────────────────────────────────────────────────────
NUM_LAYERS = 2    # 인코더/디코더 레이어 수 (참고 코드와 동일)
D_MODEL    = 256  # 임베딩 차원
NUM_HEADS  = 8    # 어텐션 헤드 수
FF_DIM     = 512  # FFN 중간 차원 (D_MODEL * 2)
DROPOUT    = 0.1  # 드롭아웃 비율

model = Transformer(
    vocab_size = VOCAB_SIZE,
    num_layers = NUM_LAYERS,
    d_model    = D_MODEL,
    num_heads  = NUM_HEADS,
    ff_dim     = FF_DIM,
    dropout    = DROPOUT
).to(device)

print(f"파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# 동작 확인
src = torch.randint(0, VOCAB_SIZE, (2, MAX_LEN)).to(device)
tgt = torch.randint(0, VOCAB_SIZE, (2, MAX_LEN)).to(device)
out = model(src, tgt)
print(f"Transformer 출력 shape: {out.shape}")  # (2, 40, 8000)

파라미터 수: 8,787,776
Transformer 출력 shape: torch.Size([2, 40, 8000])


In [16]:
# ── 정확도 계산 함수 ───────────────────────────────────────────────────────
def accuracy_fn(logits, labels):
    """
    PAD 토큰 위치는 제외하고 정확도 계산
    logits: (batch, seq_len, vocab_size)
    labels: (batch, seq_len)
    """
    preds   = logits.argmax(dim=-1)    # 가장 확률 높은 토큰 선택
    mask    = (labels != PAD_ID)       # PAD 위치 제외
    correct = (preds == labels) & mask
    return correct.float().sum() / mask.float().sum()

# ── Warmup 학습률 스케줄러 ─────────────────────────────────────────────────
def get_lr_lambda(d_model, warmup_steps=4000):
    """
    트랜스포머 원 논문 학습률 스케줄
    - warmup 구간: 학습률 선형 증가 (초반 불안정 방지)
    - warmup 이후: step^-0.5 로 서서히 감소 (과적합 방지)
    """
    d_model = float(d_model)
    def lr_lambda(step):
        step = step + 1  # 0부터 시작하므로 +1 보정
        return (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))
    return lr_lambda

# ── 옵티마이저 & 스케줄러 ──────────────────────────────────────────────────
# lr=1.0으로 설정하고 스케줄러가 실제 학습률 조정
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=get_lr_lambda(D_MODEL, warmup_steps=4000)
)
# PAD 위치는 loss 계산에서 제외
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

# ── 학습/검증 함수 ─────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, scheduler):
    model.train()
    total_loss, total_acc = 0, 0

    for enc_input, dec_input, label in loader:
        enc_input = enc_input.to(device)
        dec_input = dec_input.to(device)
        label     = label.to(device)

        # 마스크 생성
        src_mask        = create_padding_mask(enc_input)    # 인코더 PAD 마스크
        look_ahead_mask = create_look_ahead_mask(dec_input) # 디코더 미래 토큰 마스크

        optimizer.zero_grad()
        logits = model(enc_input, dec_input, src_mask, look_ahead_mask)

        # logits: (batch, seq_len, vocab_size) → CrossEntropy용 차원 변환
        loss = criterion(logits.permute(0, 2, 1), label)
        acc  = accuracy_fn(logits, label)

        loss.backward()
        # gradient clipping: 기울기 폭발 방지
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()  # 매 스텝마다 학습률 업데이트

        total_loss += loss.item()
        total_acc  += acc.item()

    return total_loss / len(loader), total_acc / len(loader)

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0, 0

    with torch.no_grad():
        for enc_input, dec_input, label in loader:
            enc_input = enc_input.to(device)
            dec_input = dec_input.to(device)
            label     = label.to(device)

            src_mask        = create_padding_mask(enc_input)
            look_ahead_mask = create_look_ahead_mask(dec_input)

            logits = model(enc_input, dec_input, src_mask, look_ahead_mask)
            loss   = criterion(logits.permute(0, 2, 1), label)
            acc    = accuracy_fn(logits, label)

            total_loss += loss.item()
            total_acc  += acc.item()

    return total_loss / len(loader), total_acc / len(loader)

# ── 학습 실행 ──────────────────────────────────────────────────────────────
EPOCHS              = 50
best_val_loss       = float('inf')
early_stop_count    = 0
EARLY_STOP_PATIENCE = 10  # 10 epoch 동안 개선 없으면 조기 종료

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scheduler)
    val_loss,   val_acc   = eval_epoch(model, val_loader, criterion)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        early_stop_count = 0
        torch.save(model.state_dict(), DATA_DIR + 'best_model.pt')
        print(f"           → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        if early_stop_count >= EARLY_STOP_PATIENCE:
            print(f"\n{EARLY_STOP_PATIENCE} epoch 개선 없음 → 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

Epoch 001 | Train Loss: 6.2612 Acc: 0.1488 | Val Loss: 5.0681 Acc: 0.2180 | LR: 0.000174
           → 모델 저장 (Val Loss: 5.0681)
Epoch 002 | Train Loss: 4.8703 Acc: 0.2274 | Val Loss: 4.7041 Acc: 0.2399 | LR: 0.000348
           → 모델 저장 (Val Loss: 4.7041)
Epoch 003 | Train Loss: 4.6105 Acc: 0.2404 | Val Loss: 4.5514 Acc: 0.2460 | LR: 0.000522
           → 모델 저장 (Val Loss: 4.5514)
Epoch 004 | Train Loss: 4.4572 Acc: 0.2459 | Val Loss: 4.4865 Acc: 0.2512 | LR: 0.000696
           → 모델 저장 (Val Loss: 4.4865)
Epoch 005 | Train Loss: 4.3407 Acc: 0.2504 | Val Loss: 4.4375 Acc: 0.2515 | LR: 0.000870
           → 모델 저장 (Val Loss: 4.4375)
Epoch 006 | Train Loss: 4.2566 Acc: 0.2543 | Val Loss: 4.3853 Acc: 0.2537 | LR: 0.000962
           → 모델 저장 (Val Loss: 4.3853)
Epoch 007 | Train Loss: 4.1603 Acc: 0.2590 | Val Loss: 4.3504 Acc: 0.2583 | LR: 0.000890
           → 모델 저장 (Val Loss: 4.3504)
Epoch 008 | Train Loss: 4.0684 Acc: 0.2646 | Val Loss: 4.3351 Acc: 0.2596 | LR: 0.000833
           → 모델 저장 (Va

In [17]:
# 모델 재초기화
model = Transformer(
    vocab_size = VOCAB_SIZE,
    num_layers = NUM_LAYERS,
    d_model    = D_MODEL,
    num_heads  = NUM_HEADS,
    ff_dim     = FF_DIM,
    dropout    = DROPOUT
).to(device)

# warmup_steps 4000 → 8000으로 늘림
# 데이터가 많아졌으므로 학습률이 더 천천히 올라가야 안정적
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=get_lr_lambda(D_MODEL, warmup_steps=8000)
)

EPOCHS              = 100
best_val_loss       = float('inf')
early_stop_count    = 0
EARLY_STOP_PATIENCE = 15

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scheduler)
    val_loss,   val_acc   = eval_epoch(model, val_loader, criterion)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        early_stop_count = 0
        torch.save(model.state_dict(), DATA_DIR + 'best_model.pt')
        print(f"           → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        if early_stop_count >= EARLY_STOP_PATIENCE:
            print(f"\n{EARLY_STOP_PATIENCE} epoch 개선 없음 → 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

Epoch 001 | Train Loss: 6.9324 Acc: 0.1173 | Val Loss: 5.4474 Acc: 0.1855 | LR: 0.000062
           → 모델 저장 (Val Loss: 5.4474)
Epoch 002 | Train Loss: 5.1763 Acc: 0.2075 | Val Loss: 4.9428 Acc: 0.2250 | LR: 0.000123
           → 모델 저장 (Val Loss: 4.9428)
Epoch 003 | Train Loss: 4.8358 Acc: 0.2294 | Val Loss: 4.7280 Acc: 0.2374 | LR: 0.000185
           → 모델 저장 (Val Loss: 4.7280)
Epoch 004 | Train Loss: 4.6559 Acc: 0.2383 | Val Loss: 4.6233 Acc: 0.2456 | LR: 0.000246
           → 모델 저장 (Val Loss: 4.6233)
Epoch 005 | Train Loss: 4.5283 Acc: 0.2444 | Val Loss: 4.5242 Acc: 0.2491 | LR: 0.000308
           → 모델 저장 (Val Loss: 4.5242)
Epoch 006 | Train Loss: 4.4248 Acc: 0.2485 | Val Loss: 4.4676 Acc: 0.2506 | LR: 0.000369
           → 모델 저장 (Val Loss: 4.4676)
Epoch 007 | Train Loss: 4.3358 Acc: 0.2522 | Val Loss: 4.4281 Acc: 0.2518 | LR: 0.000431
           → 모델 저장 (Val Loss: 4.4281)
Epoch 008 | Train Loss: 4.2528 Acc: 0.2559 | Val Loss: 4.3809 Acc: 0.2576 | LR: 0.000492
           → 모델 저장 (Va

In [18]:
# 너무 짧거나 긴 문장 제거 (노이즈 감소)
filtered_pairs = [
    (q, a) for q, a in cornell_pairs
    if 3 <= len(q.split()) <= 20   # 3~20 단어 사이 질문만
    and 3 <= len(a.split()) <= 20  # 3~20 단어 사이 답변만
]
print(f"정제 전: {len(cornell_pairs):,}개")
print(f"정제 후: {len(filtered_pairs):,}개")

정제 전: 50,000개
정제 후: 27,940개


In [20]:
# 정제된 데이터로 데이터셋 재구성
train_pairs, val_pairs = train_test_split(filtered_pairs, test_size=0.1, random_state=42)

train_dataset = CornellDataset(train_pairs, sp, MAX_LEN)
val_dataset   = CornellDataset(val_pairs,   sp, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

print(f"학습 데이터: {len(train_dataset):,}개")
print(f"검증 데이터: {len(val_dataset):,}개")

# PAD 비율 재확인 (문장 길이가 비슷해졌으니 PAD 비율도 줄어들어야 함)
sample_dec = torch.stack([train_dataset[i][1] for i in range(200)])
pad_ratio  = (sample_dec == PAD_ID).float().mean().item()
print(f"PAD 비율: {pad_ratio * 100:.1f}%")

# 모델 재초기화
model = Transformer(
    vocab_size = VOCAB_SIZE,
    num_layers = NUM_LAYERS,
    d_model    = D_MODEL,
    num_heads  = NUM_HEADS,
    ff_dim     = FF_DIM,
    dropout    = DROPOUT
).to(device)

# warmup_steps 조정
# 데이터가 줄었으므로 4000으로 다시 낮춤
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=get_lr_lambda(D_MODEL, warmup_steps=4000)
)

EPOCHS              = 100
best_val_loss       = float('inf')
early_stop_count    = 0
EARLY_STOP_PATIENCE = 15

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scheduler)
    val_loss,   val_acc   = eval_epoch(model, val_loader, criterion)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        early_stop_count = 0
        torch.save(model.state_dict(), DATA_DIR + 'best_model.pt')
        print(f"           → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        if early_stop_count >= EARLY_STOP_PATIENCE:
            print(f"\n{EARLY_STOP_PATIENCE} epoch 개선 없음 → 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

학습 데이터: 25,146개
검증 데이터: 2,794개
PAD 비율: 73.7%
Epoch 001 | Train Loss: 6.8123 Acc: 0.1440 | Val Loss: 5.2140 Acc: 0.2315 | LR: 0.000097
           → 모델 저장 (Val Loss: 5.2140)
Epoch 002 | Train Loss: 4.9141 Acc: 0.2479 | Val Loss: 4.7321 Acc: 0.2656 | LR: 0.000194
           → 모델 저장 (Val Loss: 4.7321)
Epoch 003 | Train Loss: 4.5593 Acc: 0.2728 | Val Loss: 4.5244 Acc: 0.2830 | LR: 0.000292
           → 모델 저장 (Val Loss: 4.5244)
Epoch 004 | Train Loss: 4.3543 Acc: 0.2839 | Val Loss: 4.4093 Acc: 0.2921 | LR: 0.000389
           → 모델 저장 (Val Loss: 4.4093)
Epoch 005 | Train Loss: 4.2055 Acc: 0.2904 | Val Loss: 4.3493 Acc: 0.2907 | LR: 0.000486
           → 모델 저장 (Val Loss: 4.3493)
Epoch 006 | Train Loss: 4.0839 Acc: 0.2947 | Val Loss: 4.3064 Acc: 0.2984 | LR: 0.000583
           → 모델 저장 (Val Loss: 4.3064)
Epoch 007 | Train Loss: 3.9726 Acc: 0.2994 | Val Loss: 4.2868 Acc: 0.2972 | LR: 0.000680
           → 모델 저장 (Val Loss: 4.2868)
Epoch 008 | Train Loss: 3.8764 Acc: 0.3040 | Val Loss: 4.2724 Acc:

In [3]:
!pip install sentencepiece scikit-learn -q

In [1]:
import os
import re
import math
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"디바이스: {device}")

# ── 특수 토큰 ──────────────────────────────────────────────────────────────
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

# ── 하이퍼파라미터 ─────────────────────────────────────────────────────────
DATA_DIR   = '/home/jovyan/work/transformer_chatbot/data/'
VOCAB_SIZE = 8000
MAX_LEN    = 40
BATCH_SIZE = 64
NUM_LAYERS = 2
D_MODEL    = 256
NUM_HEADS  = 8
FF_DIM     = 512
DROPOUT    = 0.1

# ── 데이터 로드 ────────────────────────────────────────────────────────────
def preprocess_en(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)
    return sentence.strip()

def load_cornell_data(path_to_lines, path_to_convs, max_samples=50000):
    id2line = {}
    with open(path_to_lines, 'r', errors='ignore') as f:
        for line in f:
            parts = line.strip().split(" +++$+++ ")
            if len(parts) >= 5:
                id2line[parts[0]] = parts[4]
    pairs = []
    with open(path_to_convs, 'r', errors='ignore') as f:
        for line in f:
            parts = line.strip().split(" +++$+++ ")
            if len(parts) < 4:
                continue
            conv_list = [c.strip("' ") for c in parts[3][1:-1].split(", ")]
            for i in range(len(conv_list) - 1):
                q = preprocess_en(id2line.get(conv_list[i],   ""))
                a = preprocess_en(id2line.get(conv_list[i+1], ""))
                if q and a:
                    pairs.append((q, a))
                if len(pairs) >= max_samples:
                    return pairs
    return pairs

cornell_dir   = DATA_DIR + 'cornell movie-dialogs corpus/'
cornell_pairs = load_cornell_data(
    cornell_dir + 'movie_lines.txt',
    cornell_dir + 'movie_conversations.txt'
)

# 문장 길이 3~20 단어로 정제
filtered_pairs = [
    (q, a) for q, a in cornell_pairs
    if 3 <= len(q.split()) <= 20
    and 3 <= len(a.split()) <= 20
]
print(f"정제 후 데이터: {len(filtered_pairs):,}개")

# ── 토크나이저 로드 ────────────────────────────────────────────────────────
sp = spm.SentencePieceProcessor()
sp.Load(DATA_DIR + 'spm_cornell.model')
print(f"토크나이저 로드 완료 | vocab: {sp.get_piece_size()}")

# ── 데이터셋 ───────────────────────────────────────────────────────────────
class CornellDataset(Dataset):
    def __init__(self, pairs, sp, max_len):
        self.data = []
        for q, a in pairs:
            enc_ids   = [BOS_ID] + sp.encode_as_ids(q) + [EOS_ID]
            enc_ids   = enc_ids[:max_len] + [PAD_ID] * (max_len - len(enc_ids[:max_len]))
            dec_ids   = [BOS_ID] + sp.encode_as_ids(a)
            dec_ids   = dec_ids[:max_len] + [PAD_ID] * (max_len - len(dec_ids[:max_len]))
            label_ids = sp.encode_as_ids(a) + [EOS_ID]
            label_ids = label_ids[:max_len] + [PAD_ID] * (max_len - len(label_ids[:max_len]))
            self.data.append((
                torch.tensor(enc_ids,   dtype=torch.long),
                torch.tensor(dec_ids,   dtype=torch.long),
                torch.tensor(label_ids, dtype=torch.long),
            ))
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

train_pairs, val_pairs = train_test_split(filtered_pairs, test_size=0.1, random_state=42)
train_dataset = CornellDataset(train_pairs, sp, MAX_LEN)
val_dataset   = CornellDataset(val_pairs,   sp, MAX_LEN)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
print(f"학습: {len(train_dataset):,}개 | 검증: {len(val_dataset):,}개")

# ── 모델 구성 ──────────────────────────────────────────────────────────────
def scaled_dot_product_attention(query, key, value, mask=None):
    d_k    = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, value), weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.depth     = d_model // num_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)
    def split_heads(self, x, b):
        return x.view(b, -1, self.num_heads, self.depth).permute(0, 2, 1, 3)
    def forward(self, q, k, v, mask=None):
        b = q.size(0)
        q, k, v = self.split_heads(self.wq(q), b), self.split_heads(self.wk(k), b), self.split_heads(self.wv(v), b)
        out, _  = scaled_dot_product_attention(q, k, v, mask)
        out     = out.permute(0, 2, 1, 3).contiguous().view(b, -1, self.num_heads * self.depth)
        return self.wo(out)

class PositionalEncoding(nn.Module):
    def __init__(self, position, d_model, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(position, d_model)
        pos = torch.arange(0, position).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model))
        self.norm1, self.norm2 = nn.LayerNorm(d_model, eps=1e-6), nn.LayerNorm(d_model, eps=1e-6)
        self.drop1, self.drop2 = nn.Dropout(dropout), nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = self.norm1(x + self.drop1(self.mha(x, x, x, mask)))
        return self.norm2(x + self.drop2(self.ffn(x)))

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        self.ffn  = nn.Sequential(nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model))
        self.norm1, self.norm2, self.norm3 = nn.LayerNorm(d_model, eps=1e-6), nn.LayerNorm(d_model, eps=1e-6), nn.LayerNorm(d_model, eps=1e-6)
        self.drop1, self.drop2, self.drop3 = nn.Dropout(dropout), nn.Dropout(dropout), nn.Dropout(dropout)
    def forward(self, x, enc_out, look_ahead_mask=None, padding_mask=None):
        x = self.norm1(x + self.drop1(self.mha1(x, x, x, look_ahead_mask)))
        x = self.norm2(x + self.drop2(self.mha2(x, enc_out, enc_out, padding_mask)))
        return self.norm3(x + self.drop3(self.ffn(x)))

class Encoder(nn.Module):
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.d_model      = d_model
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoding = PositionalEncoding(vocab_size, d_model, dropout)
        self.layers       = nn.ModuleList([EncoderLayer(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.dropout      = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = self.dropout(self.pos_encoding(self.embedding(x) * math.sqrt(self.d_model)))
        for layer in self.layers:
            x = layer(x, mask)
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.d_model      = d_model
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoding = PositionalEncoding(vocab_size, d_model, dropout)
        self.layers       = nn.ModuleList([DecoderLayer(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.dropout      = nn.Dropout(dropout)
    def forward(self, x, enc_out, look_ahead_mask=None, padding_mask=None):
        x = self.dropout(self.pos_encoding(self.embedding(x) * math.sqrt(self.d_model)))
        for layer in self.layers:
            x = layer(x, enc_out, look_ahead_mask, padding_mask)
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.encoder     = Encoder(vocab_size, num_layers, d_model, num_heads, ff_dim, dropout)
        self.decoder     = Decoder(vocab_size, num_layers, d_model, num_heads, ff_dim, dropout)
        self.final_layer = nn.Linear(d_model, vocab_size)
    def forward(self, src, tgt, src_mask=None, look_ahead_mask=None):
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt, enc_out, look_ahead_mask, src_mask)
        return self.final_layer(dec_out)

def create_padding_mask(x):
    return (x != PAD_ID).float().unsqueeze(1).unsqueeze(2)

def create_look_ahead_mask(x):
    seq_len    = x.size(1)
    look_ahead = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).unsqueeze(0).unsqueeze(1)
    padding    = create_padding_mask(x)
    return torch.min(look_ahead, padding)

model = Transformer(VOCAB_SIZE, NUM_LAYERS, D_MODEL, NUM_HEADS, FF_DIM, DROPOUT).to(device)
print(f"파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ── 학습 함수 ──────────────────────────────────────────────────────────────
def accuracy_fn(logits, labels):
    preds   = logits.argmax(dim=-1)
    mask    = (labels != PAD_ID)
    return ((preds == labels) & mask).float().sum() / mask.float().sum()

def get_lr_lambda(d_model, warmup_steps=4000):
    d_model = float(d_model)
    def lr_lambda(step):
        step = step + 1
        return (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))
    return lr_lambda

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_lambda(D_MODEL, warmup_steps=4000))

def train_epoch(model, loader, optimizer, criterion, scheduler):
    model.train()
    total_loss, total_acc = 0, 0
    for enc_input, dec_input, label in loader:
        enc_input, dec_input, label = enc_input.to(device), dec_input.to(device), label.to(device)
        src_mask        = create_padding_mask(enc_input)
        look_ahead_mask = create_look_ahead_mask(dec_input)
        optimizer.zero_grad()
        logits = model(enc_input, dec_input, src_mask, look_ahead_mask)
        loss   = criterion(logits.permute(0, 2, 1), label)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        total_acc  += accuracy_fn(logits, label).item()
    return total_loss / len(loader), total_acc / len(loader)

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0, 0
    with torch.no_grad():
        for enc_input, dec_input, label in loader:
            enc_input, dec_input, label = enc_input.to(device), dec_input.to(device), label.to(device)
            src_mask        = create_padding_mask(enc_input)
            look_ahead_mask = create_look_ahead_mask(dec_input)
            logits = model(enc_input, dec_input, src_mask, look_ahead_mask)
            total_loss += criterion(logits.permute(0, 2, 1), label).item()
            total_acc  += accuracy_fn(logits, label).item()
    return total_loss / len(loader), total_acc / len(loader)

# ── 학습 실행 ──────────────────────────────────────────────────────────────
EPOCHS, best_val_loss, early_stop_count, PATIENCE = 100, float('inf'), 0, 15

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scheduler)
    val_loss,   val_acc   = eval_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
    if val_loss < best_val_loss:
        best_val_loss, early_stop_count = val_loss, 0
        torch.save(model.state_dict(), DATA_DIR + 'best_model.pt')
        print(f"           → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        if early_stop_count >= PATIENCE:
            print(f"\n{PATIENCE} epoch 개선 없음 → 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

디바이스: cuda
정제 후 데이터: 27,940개
토크나이저 로드 완료 | vocab: 8000
학습: 25,146개 | 검증: 2,794개
파라미터 수: 8,787,776
Epoch 001 | Train Loss: 6.8554 Acc: 0.1516 | Val Loss: 5.2368 Acc: 0.2307 | LR: 0.000097
           → 모델 저장 (Val Loss: 5.2368)
Epoch 002 | Train Loss: 4.9198 Acc: 0.2488 | Val Loss: 4.7208 Acc: 0.2715 | LR: 0.000194
           → 모델 저장 (Val Loss: 4.7208)
Epoch 003 | Train Loss: 4.5596 Acc: 0.2735 | Val Loss: 4.5199 Acc: 0.2859 | LR: 0.000292
           → 모델 저장 (Val Loss: 4.5199)
Epoch 004 | Train Loss: 4.3567 Acc: 0.2841 | Val Loss: 4.4127 Acc: 0.2901 | LR: 0.000389
           → 모델 저장 (Val Loss: 4.4127)
Epoch 005 | Train Loss: 4.2101 Acc: 0.2901 | Val Loss: 4.3699 Acc: 0.2946 | LR: 0.000486
           → 모델 저장 (Val Loss: 4.3699)
Epoch 006 | Train Loss: 4.0897 Acc: 0.2946 | Val Loss: 4.3194 Acc: 0.2958 | LR: 0.000583
           → 모델 저장 (Val Loss: 4.3194)
Epoch 007 | Train Loss: 3.9787 Acc: 0.2985 | Val Loss: 4.2836 Acc: 0.2985 | LR: 0.000680
           → 모델 저장 (Val Loss: 4.2836)
Epoch 008 | T

In [2]:
# ── 한국어 데이터 로드 및 전처리 ──────────────────────────────────────────
df = pd.read_csv(DATA_DIR + 'ChatbotData.csv')
df = df.dropna()

def preprocess_ko(text):
    # 한글, 영문, 숫자, 기본 구두점만 유지
    text = re.sub(r"[^가-힣a-zA-Z0-9?.!,\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['Q'] = df['Q'].apply(preprocess_ko)
df['A'] = df['A'].apply(preprocess_ko)
df = df[(df['Q'] != '') & (df['A'] != '')]
print(f"한국어 데이터 수: {len(df):,}개")
print(df.head(3))

# ── 한국어 SentencePiece 토크나이저 학습 ──────────────────────────────────
ko_corpus_path = DATA_DIR + 'ko_corpus.txt'
with open(ko_corpus_path, 'w', encoding='utf-8') as f:
    for text in pd.concat([df['Q'], df['A']]):
        f.write(text.strip() + '\n')

spm.SentencePieceTrainer.Train(
    input              = ko_corpus_path,
    model_prefix       = DATA_DIR + 'spm_korean',
    vocab_size         = 8000,
    character_coverage = 0.9995,  # 한국어는 글자 수가 많아서 0.9995 권장
    model_type         = 'bpe',
    pad_id=PAD_ID, bos_id=BOS_ID, eos_id=EOS_ID, unk_id=UNK_ID,
    pad_piece='[PAD]', bos_piece='[BOS]', eos_piece='[EOS]', unk_piece='[UNK]',
)

# 한국어 토크나이저 로드
sp_ko = spm.SentencePieceProcessor()
sp_ko.Load(DATA_DIR + 'spm_korean.model')

# 동작 확인
test = "오늘 기분이 너무 안좋아"
print(f"\n원문: {test}")
print(f"토크나이징: {sp_ko.encode_as_pieces(test)}")
print(f"vocab 크기: {sp_ko.get_piece_size()}")

한국어 데이터 수: 11,823개
              Q            A  label
0        12시 땡!   하루가 또 가네요.      0
1   1지망 학교 떨어졌어    위로해 드립니다.      0
2  3박4일 놀러가고 싶다  여행은 언제나 좋죠.      0

원문: 오늘 기분이 너무 안좋아
토크나이징: ['▁오늘', '▁기분이', '▁너무', '▁안좋아']
vocab 크기: 8000


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /home/jovyan/work/transformer_chatbot/data/ko_corpus.txt
  input_format: 
  model_prefix: /home/jovyan/work/transformer_chatbot/data/spm_korean
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: [UNK]
  bos_piece: [BOS]
  eos_piec

In [3]:
# ── 한국어 데이터셋 구성 ───────────────────────────────────────────────────
# MAX_LEN을 25로 줄임
# 한국어 답변 평균 길이가 5.7토큰으로 짧으므로
# 40으로 설정하면 PAD 비율이 너무 높아져서 EOS만 예측하는 문제 발생
MAX_LEN = 25

class KoreanDataset(Dataset):
    def __init__(self, df, sp, max_len):
        self.data = []
        for _, row in df.iterrows():
            q, a = row['Q'], row['A']

            enc_ids   = [BOS_ID] + sp.encode_as_ids(q) + [EOS_ID]
            enc_ids   = enc_ids[:max_len] + [PAD_ID] * (max_len - len(enc_ids[:max_len]))

            dec_ids   = [BOS_ID] + sp.encode_as_ids(a)
            dec_ids   = dec_ids[:max_len] + [PAD_ID] * (max_len - len(dec_ids[:max_len]))

            label_ids = sp.encode_as_ids(a) + [EOS_ID]
            label_ids = label_ids[:max_len] + [PAD_ID] * (max_len - len(label_ids[:max_len]))

            self.data.append((
                torch.tensor(enc_ids,   dtype=torch.long),
                torch.tensor(dec_ids,   dtype=torch.long),
                torch.tensor(label_ids, dtype=torch.long),
            ))

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

# 학습/검증 분리
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset_ko = KoreanDataset(train_df, sp_ko, MAX_LEN)
val_dataset_ko   = KoreanDataset(val_df,   sp_ko, MAX_LEN)
train_loader_ko  = DataLoader(train_dataset_ko, batch_size=BATCH_SIZE, shuffle=True)
val_loader_ko    = DataLoader(val_dataset_ko,   batch_size=BATCH_SIZE)

print(f"학습 데이터: {len(train_dataset_ko):,}개")
print(f"검증 데이터: {len(val_dataset_ko):,}개")

# PAD 비율 확인
sample = torch.stack([train_dataset_ko[i][1] for i in range(200)])
print(f"PAD 비율: {(sample == PAD_ID).float().mean().item() * 100:.1f}%")

# ── 한국어 모델 초기화 ─────────────────────────────────────────────────────
# MAX_LEN이 바뀌었으므로 모델 새로 초기화
model_ko = Transformer(
    vocab_size = VOCAB_SIZE,
    num_layers = NUM_LAYERS,
    d_model    = D_MODEL,
    num_heads  = NUM_HEADS,
    ff_dim     = FF_DIM,
    dropout    = DROPOUT
).to(device)

print(f"파라미터 수: {sum(p.numel() for p in model_ko.parameters() if p.requires_grad):,}")

# ── 옵티마이저 & 스케줄러 ──────────────────────────────────────────────────
# warmup_steps=4000: 한국어 데이터는 11,823개로 적으므로 4000이 적합
optimizer_ko = optim.Adam(model_ko.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler_ko = lr_scheduler.LambdaLR(
    optimizer_ko,
    lr_lambda=get_lr_lambda(D_MODEL, warmup_steps=4000)
)

# ── 학습 실행 ──────────────────────────────────────────────────────────────
EPOCHS, best_val_loss, early_stop_count, PATIENCE = 100, float('inf'), 0, 10

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model_ko, train_loader_ko, optimizer_ko, criterion, scheduler_ko)
    val_loss,   val_acc   = eval_epoch(model_ko, val_loader_ko, criterion)

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {optimizer_ko.param_groups[0]['lr']:.6f}")

    if val_loss < best_val_loss:
        best_val_loss, early_stop_count = val_loss, 0
        torch.save(model_ko.state_dict(), DATA_DIR + 'best_model_ko.pt')
        print(f"           → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        if early_stop_count >= PATIENCE:
            print(f"\n{PATIENCE} epoch 개선 없음 → 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

학습 데이터: 10,640개
검증 데이터: 1,183개
PAD 비율: 72.8%
파라미터 수: 8,787,776
Epoch 001 | Train Loss: 7.9606 Acc: 0.1805 | Val Loss: 6.6262 Acc: 0.2910 | LR: 0.000042
           → 모델 저장 (Val Loss: 6.6262)
Epoch 002 | Train Loss: 6.2314 Acc: 0.2885 | Val Loss: 5.9158 Acc: 0.2917 | LR: 0.000083
           → 모델 저장 (Val Loss: 5.9158)
Epoch 003 | Train Loss: 5.7598 Acc: 0.2939 | Val Loss: 5.6762 Acc: 0.3039 | LR: 0.000124
           → 모델 저장 (Val Loss: 5.6762)
Epoch 004 | Train Loss: 5.5128 Acc: 0.3044 | Val Loss: 5.4710 Acc: 0.3094 | LR: 0.000165
           → 모델 저장 (Val Loss: 5.4710)
Epoch 005 | Train Loss: 5.2745 Acc: 0.3133 | Val Loss: 5.2692 Acc: 0.3234 | LR: 0.000207
           → 모델 저장 (Val Loss: 5.2692)
Epoch 006 | Train Loss: 5.0063 Acc: 0.3256 | Val Loss: 5.0880 Acc: 0.3354 | LR: 0.000248
           → 모델 저장 (Val Loss: 5.0880)
Epoch 007 | Train Loss: 4.7457 Acc: 0.3382 | Val Loss: 4.9033 Acc: 0.3476 | LR: 0.000289
           → 모델 저장 (Val Loss: 4.9033)
Epoch 008 | Train Loss: 4.4714 Acc: 0.3549 | Val

In [4]:
# 최적 모델 로드
model_ko.load_state_dict(torch.load(DATA_DIR + 'best_model_ko.pt'))
print("모델 로드 완료")

# ── 예측 함수 ──────────────────────────────────────────────────────────────
# 참고 코드와 동일하게 Greedy Decoding 방식 사용
# argmax로 가장 확률 높은 토큰을 순서대로 선택
def predict(sentence, model, sp, max_len=MAX_LEN):
    model.eval()

    # 입력 전처리 및 인코더 입력 구성
    sentence = preprocess_ko(sentence)
    enc_ids  = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    enc_ids  = enc_ids[:max_len] + [PAD_ID] * (max_len - len(enc_ids[:max_len]))
    src      = torch.tensor([enc_ids], dtype=torch.long).to(device)

    # 인코더 출력 계산 (한 번만 실행)
    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_out = model_ko.encoder(src, src_mask)

    # 디코더 입력을 BOS 토큰으로 시작해서 토큰을 하나씩 생성
    dec_ids = [BOS_ID]

    for _ in range(max_len):
        tgt             = torch.tensor([dec_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)

        with torch.no_grad():
            dec_out = model_ko.decoder(tgt, enc_out, look_ahead_mask, src_mask)
            # 마지막 타임스텝의 logit으로 다음 토큰 예측
            logits  = model_ko.final_layer(dec_out[:, -1, :])
            # argmax: 가장 확률 높은 토큰 선택 (Greedy Decoding)
            next_id = logits.argmax(dim=-1).item()

        # EOS 토큰이 나오면 생성 종료
        if next_id == EOS_ID:
            break

        dec_ids.append(next_id)

    # BOS 제외하고 디코딩
    return sp.decode_ids(dec_ids[1:])

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "친구가 나를 무시해",
    "요즘 잠을 못자겠어",
]

for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict(t, model_ko, sp_ko)}")
    print()

모델 로드 완료
Q: 오늘 기분이 너무 안좋아
A: 조심해야해요.

Q: 나 요즘 너무 외로워
A: 좀 더 편하게 생각해보세요.

Q: 밥은 먹었어?
A: 저는 배터리가 밥이예요.

Q: 취업이 너무 힘들어
A: 그런 감이 없지 않죠.

Q: 사랑이 뭔지 모르겠어
A: 사랑은 변화하는 거예요.

Q: 친구가 나를 무시해
A: 친구를 사귀어 보세요.

Q: 요즘 잠을 못자겠어
A: 잠이 최고의 보약이에요.



In [5]:
# 모델 재초기화
model_ko = Transformer(
    vocab_size = VOCAB_SIZE,
    num_layers = NUM_LAYERS,
    d_model    = D_MODEL,
    num_heads  = NUM_HEADS,
    ff_dim     = FF_DIM,
    dropout    = DROPOUT
).to(device)

# warmup_steps 4000 → 2000으로 줄임
# 데이터가 적으므로 더 빠르게 최적 학습률에 도달하도록
# 에폭도 200으로 늘려서 충분히 학습할 기회를 줌
optimizer_ko = optim.Adam(model_ko.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler_ko = lr_scheduler.LambdaLR(
    optimizer_ko,
    lr_lambda=get_lr_lambda(D_MODEL, warmup_steps=2000)
)

EPOCHS, best_val_loss, early_stop_count, PATIENCE = 200, float('inf'), 0, 15

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model_ko, train_loader_ko, optimizer_ko, criterion, scheduler_ko)
    val_loss,   val_acc   = eval_epoch(model_ko, val_loader_ko, criterion)

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {optimizer_ko.param_groups[0]['lr']:.6f}")

    if val_loss < best_val_loss:
        best_val_loss, early_stop_count = val_loss, 0
        torch.save(model_ko.state_dict(), DATA_DIR + 'best_model_ko.pt')
        print(f"           → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        if early_stop_count >= PATIENCE:
            print(f"\n{PATIENCE} epoch 개선 없음 → 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

Epoch 001 | Train Loss: 7.3329 Acc: 0.2132 | Val Loss: 6.0516 Acc: 0.2915 | LR: 0.000117
           → 모델 저장 (Val Loss: 6.0516)
Epoch 002 | Train Loss: 5.7960 Acc: 0.2952 | Val Loss: 5.6311 Acc: 0.3063 | LR: 0.000234
           → 모델 저장 (Val Loss: 5.6311)
Epoch 003 | Train Loss: 5.4191 Acc: 0.3093 | Val Loss: 5.3228 Acc: 0.3212 | LR: 0.000351
           → 모델 저장 (Val Loss: 5.3228)
Epoch 004 | Train Loss: 5.0144 Acc: 0.3253 | Val Loss: 5.0083 Acc: 0.3365 | LR: 0.000467
           → 모델 저장 (Val Loss: 5.0083)
Epoch 005 | Train Loss: 4.5868 Acc: 0.3475 | Val Loss: 4.7150 Acc: 0.3597 | LR: 0.000584
           → 모델 저장 (Val Loss: 4.7150)
Epoch 006 | Train Loss: 4.1413 Acc: 0.3783 | Val Loss: 4.4790 Acc: 0.3794 | LR: 0.000701
           → 모델 저장 (Val Loss: 4.4790)
Epoch 007 | Train Loss: 3.6896 Acc: 0.4141 | Val Loss: 4.2880 Acc: 0.3945 | LR: 0.000818
           → 모델 저장 (Val Loss: 4.2880)
Epoch 008 | Train Loss: 3.2308 Acc: 0.4569 | Val Loss: 4.1319 Acc: 0.4109 | LR: 0.000934
           → 모델 저장 (Va

In [6]:
# 모델 크기 키우기
# NUM_LAYERS 2→4, D_MODEL 256→512, FF_DIM 512→1024
# 파라미터가 많아질수록 더 복잡한 패턴 학습 가능
NUM_LAYERS_BIG = 4
D_MODEL_BIG    = 512
NUM_HEADS_BIG  = 8    # D_MODEL/NUM_HEADS = 512/8 = 64 (depth per head)
FF_DIM_BIG     = 1024
DROPOUT_BIG    = 0.2  # 모델이 커졌으니 dropout도 높여서 과적합 억제

model_big = Transformer(
    vocab_size = VOCAB_SIZE,
    num_layers = NUM_LAYERS_BIG,
    d_model    = D_MODEL_BIG,
    num_heads  = NUM_HEADS_BIG,
    ff_dim     = FF_DIM_BIG,
    dropout    = DROPOUT_BIG
).to(device)

print(f"파라미터 수: {sum(p.numel() for p in model_big.parameters() if p.requires_grad):,}")

# warmup_steps=4000으로 다시 원복
# 모델이 커졌으니 안정적인 학습률 스케줄 필요
optimizer_big = optim.Adam(model_big.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler_big = lr_scheduler.LambdaLR(
    optimizer_big,
    lr_lambda=get_lr_lambda(D_MODEL_BIG, warmup_steps=4000)
)

EPOCHS, best_val_loss, early_stop_count, PATIENCE = 200, float('inf'), 0, 15

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model_big, train_loader_ko, optimizer_big, criterion, scheduler_big)
    val_loss,   val_acc   = eval_epoch(model_big, val_loader_ko, criterion)

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {optimizer_big.param_groups[0]['lr']:.6f}")

    if val_loss < best_val_loss:
        best_val_loss, early_stop_count = val_loss, 0
        torch.save(model_big.state_dict(), DATA_DIR + 'best_model_big.pt')
        print(f"           → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        if early_stop_count >= PATIENCE:
            print(f"\n{PATIENCE} epoch 개선 없음 → 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

파라미터 수: 33,324,864
Epoch 001 | Train Loss: 7.3166 Acc: 0.2286 | Val Loss: 6.2526 Acc: 0.2912 | LR: 0.000029
           → 모델 저장 (Val Loss: 6.2526)
Epoch 002 | Train Loss: 5.9931 Acc: 0.2921 | Val Loss: 5.7737 Acc: 0.2958 | LR: 0.000059
           → 모델 저장 (Val Loss: 5.7737)
Epoch 003 | Train Loss: 5.6389 Acc: 0.3024 | Val Loss: 5.5283 Acc: 0.3120 | LR: 0.000088
           → 모델 저장 (Val Loss: 5.5283)
Epoch 004 | Train Loss: 5.3896 Acc: 0.3115 | Val Loss: 5.2954 Acc: 0.3217 | LR: 0.000117
           → 모델 저장 (Val Loss: 5.2954)
Epoch 005 | Train Loss: 5.1113 Acc: 0.3233 | Val Loss: 5.0722 Acc: 0.3356 | LR: 0.000146
           → 모델 저장 (Val Loss: 5.0722)
Epoch 006 | Train Loss: 4.8150 Acc: 0.3385 | Val Loss: 4.8538 Acc: 0.3505 | LR: 0.000175
           → 모델 저장 (Val Loss: 4.8538)
Epoch 007 | Train Loss: 4.5318 Acc: 0.3555 | Val Loss: 4.6890 Acc: 0.3623 | LR: 0.000204
           → 모델 저장 (Val Loss: 4.6890)
Epoch 008 | Train Loss: 4.2517 Acc: 0.3735 | Val Loss: 4.5000 Acc: 0.3788 | LR: 0.000234
   

In [7]:
# 최적 모델 로드
model_big.load_state_dict(torch.load(DATA_DIR + 'best_model_big.pt'))
print("모델 로드 완료")

def predict_big(sentence, model, sp, max_len=MAX_LEN):
    model.eval()
    sentence = preprocess_ko(sentence)
    enc_ids  = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    enc_ids  = enc_ids[:max_len] + [PAD_ID] * (max_len - len(enc_ids[:max_len]))
    src      = torch.tensor([enc_ids], dtype=torch.long).to(device)

    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_out = model.encoder(src, src_mask)

    dec_ids = [BOS_ID]
    for _ in range(max_len):
        tgt             = torch.tensor([dec_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)
        with torch.no_grad():
            dec_out = model.decoder(tgt, enc_out, look_ahead_mask, src_mask)
            next_id = model.final_layer(dec_out[:, -1, :]).argmax(dim=-1).item()
        if next_id == EOS_ID:
            break
        dec_ids.append(next_id)

    return sp.decode_ids(dec_ids[1:])

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "친구가 나를 무시해",
    "요즘 잠을 못자겠어",
]

for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict_big(t, model_big, sp_ko)}")
    print()

모델 로드 완료
Q: 오늘 기분이 너무 안좋아
A: 오늘 미세먼지가 많데요.

Q: 나 요즘 너무 외로워
A: 외로우니까 사람이다.

Q: 밥은 먹었어?
A: 저는 배터리가 밥이예요.

Q: 취업이 너무 힘들어
A: 전생에 나라를 구하셨나요.

Q: 사랑이 뭔지 모르겠어
A: 사랑은 유지하는 게 중요한데 대단하네요.

Q: 친구가 나를 무시해
A: 친구 사귀는 곳이 아니에요.

Q: 요즘 잠을 못자겠어
A: 잠이 최고의 보약이에요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요



In [8]:
def predict_big(sentence, model, sp, max_len=MAX_LEN):
    model.eval()
    sentence = preprocess_ko(sentence)
    enc_ids  = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    enc_ids  = enc_ids[:max_len] + [PAD_ID] * (max_len - len(enc_ids[:max_len]))
    src      = torch.tensor([enc_ids], dtype=torch.long).to(device)

    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_out = model.encoder(src, src_mask)

    dec_ids       = [BOS_ID]
    recent_window = 3  # 최근 3개 토큰은 다시 못 뽑게 해서 반복 방지

    for _ in range(max_len):
        tgt             = torch.tensor([dec_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)

        with torch.no_grad():
            dec_out = model.decoder(tgt, enc_out, look_ahead_mask, src_mask)
            logits  = model.final_layer(dec_out[:, -1, :])

            # 특수 토큰 차단
            logits[0, PAD_ID] = -1e9
            logits[0, BOS_ID] = -1e9
            logits[0, UNK_ID] = -1e9

            # 최근 window 내 토큰 차단 (반복 방지)
            for token_id in dec_ids[-recent_window:]:
                logits[0, token_id] = -1e9

            next_id = logits.argmax(dim=-1).item()

        if next_id == EOS_ID:
            break

        dec_ids.append(next_id)

    return sp.decode_ids(dec_ids[1:])

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "친구가 나를 무시해",
    "요즘 잠을 못자겠어",
]

for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict_big(t, model_big, sp_ko)}")
    print()

Q: 오늘 기분이 너무 안좋아
A: 오늘 미세먼지가 많데요.

Q: 나 요즘 너무 외로워
A: 외로우니까 사람이다.

Q: 밥은 먹었어?
A: 저는 배터리가 밥이예요.

Q: 취업이 너무 힘들어
A: 전생에 나라를 구하셨나요.

Q: 사랑이 뭔지 모르겠어
A: 사랑은 유지하는 게 중요한데 대단하네요.

Q: 친구가 나를 무시해
A: 친구 사귀는 곳이 아니에요.

Q: 요즘 잠을 못자겠어
A: 잠이 최고의 보약이에요. 노력해보세요? 눈을 붙이세요. 노력해보세요?



In [9]:
def predict_big(sentence, model, sp, max_len=MAX_LEN,
                repetition_penalty=1.5,  # 이미 생성한 토큰 확률 낮춤
                recent_window=3):        # 최근 n개 토큰 완전 차단
    model.eval()
    sentence = preprocess_ko(sentence)
    enc_ids  = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    enc_ids  = enc_ids[:max_len] + [PAD_ID] * (max_len - len(enc_ids[:max_len]))
    src      = torch.tensor([enc_ids], dtype=torch.long).to(device)

    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_out = model.encoder(src, src_mask)

    dec_ids = [BOS_ID]

    for step in range(max_len):
        tgt             = torch.tensor([dec_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)

        with torch.no_grad():
            dec_out = model.decoder(tgt, enc_out, look_ahead_mask, src_mask)
            logits  = model.final_layer(dec_out[:, -1, :])

            # 특수 토큰 완전 차단
            logits[0, PAD_ID] = -1e9
            logits[0, BOS_ID] = -1e9
            logits[0, UNK_ID] = -1e9

            # 전체 생성 토큰에 반복 패널티 적용
            # 이미 나온 토큰은 확률을 낮춰서 다시 나오기 어렵게 함
            for token_id in set(dec_ids):
                logits[0, token_id] /= repetition_penalty

            # 최근 window 토큰은 완전 차단 (강한 반복 방지)
            for token_id in dec_ids[-recent_window:]:
                logits[0, token_id] = -1e9

            next_id = logits.argmax(dim=-1).item()

        if next_id == EOS_ID:
            break

        dec_ids.append(next_id)

    return sp.decode_ids(dec_ids[1:])

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "친구가 나를 무시해",
    "요즘 잠을 못자겠어",
    "나 오늘 시험 망했어",
    "남자친구가 나를 차버렸어",
    "돈이 없어서 너무 힘들어",
]

for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict_big(t, model_big, sp_ko)}")
    print()

Q: 오늘 기분이 너무 안좋아
A: 오늘 미세먼지가 많데요.

Q: 나 요즘 너무 외로워
A: 외로우니까 사람이다.

Q: 밥은 먹었어?
A: 저는 배터리가 밥이예요.

Q: 취업이 너무 힘들어
A: 전생에 나라를 구하셨나요.

Q: 사랑이 뭔지 모르겠어
A: 사랑은 유지하는 게 중요한데 대단하네요.

Q: 친구가 나를 무시해
A: 친구 사귀는 곳이 아니에요.

Q: 요즘 잠을 못자겠어
A: 잠이 최고의 보약이에요. 노력해보세요? 눈을 붙이세요. 노력해보세요!

Q: 나 오늘 시험 망했어
A: 나중에 없애주세요.

Q: 남자친구가 나를 차버렸어
A: 당신이 너무 귀여운가봐요.

Q: 돈이 없어서 너무 힘들어
A: 그래도 취미 생활이 있어야 사는 재미가 있죠.



In [13]:
# 프로젝트 회고 (Retrospective)
# 최종 Best Val Loss: 3.5625

# [Step 1] 데이터 전처리
# - 한글, 영문, 숫자, 구두점만 남기고 나머지 특수문자 제거
# - SentencePiece로 토크나이징 (단어를 숫자 ID로 변환)
#   예) "오늘 기분이" → [126, 934]
# - 모델 입력 형태로 변환
#   인코더 입력(질문): [시작] + 질문 토큰 + [끝] + [패딩]
#   디코더 입력(답변): [시작] + 답변 토큰 + [패딩]
#   정답 레이블:       답변 토큰 + [끝] + [패딩]

# [Step 2] 트랜스포머 모델 구현
# - Scaled Dot-Product Attention
#   : Q, K, V 행렬로 각 단어가 다른 단어에 얼마나 집중할지 계산
# - Multi-Head Attention
#   : 어텐션을 8개로 나눠서 병렬로 계산 (다양한 관점 학습)
# - Positional Encoding
#   : 단어의 순서 정보를 sin/cos 함수로 임베딩에 추가
# - Encoder
#   : 질문 문장을 이해해서 의미 벡터로 변환
# - Decoder
#   : 인코더 출력을 참고하며 답변 토큰을 하나씩 생성
# - 최종 모델 크기: 4개 레이어, d_model=512, 파라미터 약 3,300만 개

# [Step 3] 학습
# - 손실 함수: CrossEntropyLoss (PAD 토큰 위치는 계산 제외)
# - 옵티마이저: Adam
# - 학습률 스케줄: Warmup 방식 (처음엔 천천히, 이후 서서히 감소)
# - Early Stopping: 15번 연속 개선 없으면 자동 종료
# - 최적 모델: 18번째 에폭에서 저장

# [문제 1] 모델이 "EOS(문장 끝)" 토큰만 계속 뽑는 문제
# - 원인: 전체 입력의 79%가 패딩이라서 모델이
#         "그냥 끝내버리는 게 정답"이라고 학습해버림
# - 해결: MAX_LEN을 40 → 25로 줄여서 패딩 비율을 낮춤

# [문제 2] "골 골 골", "노력해보세요 노력해보세요" 같은 반복 생성
# - 원인: Greedy Decoding은 항상 확률 1위 토큰만 뽑기 때문에
#         한번 특정 토큰이 1위가 되면 계속 같은 토큰만 나옴
# - 해결: 최근에 나온 토큰은 다시 못 뽑게 차단하고
#         이미 나온 토큰의 확률을 패널티로 낮춤

# [문제 3] 과적합 (학습 데이터는 잘 맞추는데 새 문장은 이상한 답변)
# - 원인: 데이터가 11,823개로 너무 적음
# - 해결: dropout 높이기(0.1→0.2), Early Stopping으로 최적 시점 포착

# [문제 4] 학습률을 잘못 설정하면 성능이 오히려 나빠짐
# - warmup_steps를 4000→2000으로 줄였더니 Val Loss가 3.8→3.9로 악화
# - 원인: 학습률이 너무 빠르게 올라가면 초반부터 과적합 발생
# - 해결: warmup_steps=4000 유지

# ── 챗봇 결과 ────────────────────────────────────────────
# Q: 오늘 기분이 너무 안좋아  → A: 오늘 미세먼지가 많데요.
# Q: 나 요즘 너무 외로워      → A: 외로우니까 사람이다.
# Q: 밥은 먹었어?             → A: 저는 배터리가 밥이예요.
# Q: 취업이 너무 힘들어        → A: 전생에 나라를 구하셨나요.
# Q: 남자친구가 나를 차버렸어   → A: 당신이 너무 귀여운가봐요.
# Q: 요즘 잠을 못자겠어        → A: 잠이 최고의 보약이에요.

# [아쉬운 점]
# - 데이터가 11,823개로 적어서 맥락이 안 맞는 답변이 가끔 나옴
# - 반복 토큰 문제가 완전히 해결되지는 않음
# - 영화 대화 기반의 코넬 데이터(영어 50,000개)도 시도해봤지만
#   Val Loss 4.2 수준으로 한국어 챗봇 데이터보다 성능이 낮았음

# [개선 사]
# 1. KoGPT2 파인튜닝: 이미 수십억 개 텍스트로 학습된 모델을
#    우리 데이터에 맞게 추가 학습시키면 훨씬 자연스러운 답변 가능
# 2. 데이터 확장: AIHub 한국어 대화 데이터(100만 건 이상) 활용
# 3. Beam Search 적용: 여러 후보 경로를 탐색해서
#    더 자연스러운 문장 선택

print("=" * 30)
print("최종 모델 성능 요약")
print("=" * 30)
print(f"모델 구조    : Transformer (Encoder 4층 + Decoder 4층)")
print(f"파라미터 수  : 33,324,864개 (약 3,300만 개)")
print(f"Best Val Loss: 3.5625 (18번째 에폭)")
print(f"학습 데이터  : 10,640개")
print(f"검증 데이터  : 1,183개")
print("=" * 30)

# 최종 챗봇 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "사랑이 뭔지 모르겠어",
    "친구가 나를 무시해",
    "요즘 잠을 못자겠어",
    "나 오늘 시험 망했어",
    "남자친구가 나를 차버렸어",
    "돈이 없어서 너무 힘들어",
]

print("\n[최종 챗봇 테스트]")
for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict_big(t, model_big, sp_ko)}")
    print()

최종 모델 성능 요약
모델 구조    : Transformer (Encoder 4층 + Decoder 4층)
파라미터 수  : 33,324,864개 (약 3,300만 개)
Best Val Loss: 3.5625 (18번째 에폭)
학습 데이터  : 10,640개
검증 데이터  : 1,183개

[최종 챗봇 테스트]
Q: 오늘 기분이 너무 안좋아
A: 오늘 미세먼지가 많데요.

Q: 나 요즘 너무 외로워
A: 외로우니까 사람이다.

Q: 밥은 먹었어?
A: 저는 배터리가 밥이예요.

Q: 사랑이 뭔지 모르겠어
A: 사랑은 유지하는 게 중요한데 대단하네요.

Q: 친구가 나를 무시해
A: 친구 사귀는 곳이 아니에요.

Q: 요즘 잠을 못자겠어
A: 잠이 최고의 보약이에요. 노력해보세요? 눈을 붙이세요. 노력해보세요!

Q: 나 오늘 시험 망했어
A: 나중에 없애주세요.

Q: 남자친구가 나를 차버렸어
A: 당신이 너무 귀여운가봐요.

Q: 돈이 없어서 너무 힘들어
A: 그래도 취미 생활이 있어야 사는 재미가 있죠.



In [14]:
# 반복 패널티 없이 순수 Greedy Decoding으로 예측
# 참고 코드의 decoder_inference 함수와 동일한 방식
def predict_greedy(sentence, model, sp, max_len=MAX_LEN):
    model.eval()
    sentence = preprocess_ko(sentence)
    enc_ids  = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    enc_ids  = enc_ids[:max_len] + [PAD_ID] * (max_len - len(enc_ids[:max_len]))
    src      = torch.tensor([enc_ids], dtype=torch.long).to(device)

    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_out = model.encoder(src, src_mask)

    dec_ids = [BOS_ID]

    for _ in range(max_len):
        tgt             = torch.tensor([dec_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)

        with torch.no_grad():
            dec_out = model.decoder(tgt, enc_out, look_ahead_mask, src_mask)
            # 아무 패널티 없이 그냥 확률 1위 토큰 선택
            next_id = model.final_layer(dec_out[:, -1, :]).argmax(dim=-1).item()

        if next_id == EOS_ID:
            break

        dec_ids.append(next_id)

    return sp.decode_ids(dec_ids[1:])

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "친구가 나를 무시해",
    "요즘 잠을 못자겠어",
    "나 오늘 시험 망했어",
    "남자친구가 나를 차버렸어",
    "돈이 없어서 너무 힘들어",
]

print("[순수 Greedy 예측 결과]")
for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict_greedy(t, model_big, sp_ko)}")
    print()


[순수 Greedy 예측 결과]
Q: 오늘 기분이 너무 안좋아
A: 오늘 미세먼지가 많데요.

Q: 나 요즘 너무 외로워
A: 외로우니까 사람이다.

Q: 밥은 먹었어?
A: 저는 배터리가 밥이예요.

Q: 취업이 너무 힘들어
A: 전생에 나라를 구하셨나요.

Q: 사랑이 뭔지 모르겠어
A: 사랑은 유지하는 게 중요한데 대단하네요.

Q: 친구가 나를 무시해
A: 친구 사귀는 곳이 아니에요.

Q: 요즘 잠을 못자겠어
A: 잠이 최고의 보약이에요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요. 노력해보세요

Q: 나 오늘 시험 망했어
A: 나중에 없애주세요.

Q: 남자친구가 나를 차버렸어
A: 당신이 너무 귀여운가봐요.

Q: 돈이 없어서 너무 힘들어
A: 그래도 취미 생활이 있어야 사는 재미가 있죠.



In [16]:
# 모델 크기 줄이기
# 큰 모델(4층, d_model=512)은 데이터 대비 파라미터가 너무 많아서
# 학습 데이터를 외워버리는 과적합이 발생함
# 작은 모델(2층, d_model=128)로 줄여서 과적합 억제

NUM_LAYERS_SMALL = 2    # 4 → 2
D_MODEL_SMALL    = 128  # 512 → 128
NUM_HEADS_SMALL  = 8    # depth = 128/8 = 16
FF_DIM_SMALL     = 256  # 1024 → 256
DROPOUT_SMALL    = 0.1  # 작은 모델이니 dropout도 낮춤

model_small = Transformer(
    vocab_size = VOCAB_SIZE,
    num_layers = NUM_LAYERS_SMALL,
    d_model    = D_MODEL_SMALL,
    num_heads  = NUM_HEADS_SMALL,
    ff_dim     = FF_DIM_SMALL,
    dropout    = DROPOUT_SMALL
).to(device)

print(f"파라미터 수: {sum(p.numel() for p in model_small.parameters() if p.requires_grad):,}")

optimizer_small = optim.Adam(model_small.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler_small = lr_scheduler.LambdaLR(
    optimizer_small,
    lr_lambda=get_lr_lambda(D_MODEL_SMALL, warmup_steps=4000)
)

EPOCHS, best_val_loss, early_stop_count, PATIENCE = 200, float('inf'), 0, 15

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model_small, train_loader_ko, optimizer_small, criterion, scheduler_small)
    val_loss,   val_acc   = eval_epoch(model_small, val_loader_ko, criterion)

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {optimizer_small.param_groups[0]['lr']:.6f}")

    if val_loss < best_val_loss:
        best_val_loss, early_stop_count = val_loss, 0
        torch.save(model_small.state_dict(), DATA_DIR + 'best_model_small.pt')
        print(f"           → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        if early_stop_count >= PATIENCE:
            print(f"\n{PATIENCE} epoch 개선 없음 → 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

파라미터 수: 3,742,528
Epoch 001 | Train Loss: 8.5573 Acc: 0.0936 | Val Loss: 7.4364 Acc: 0.2913 | LR: 0.000059
           → 모델 저장 (Val Loss: 7.4364)
Epoch 002 | Train Loss: 6.7759 Acc: 0.2884 | Val Loss: 6.2134 Acc: 0.2913 | LR: 0.000117
           → 모델 저장 (Val Loss: 6.2134)
Epoch 003 | Train Loss: 5.9854 Acc: 0.2904 | Val Loss: 5.8586 Acc: 0.2925 | LR: 0.000175
           → 모델 저장 (Val Loss: 5.8586)
Epoch 004 | Train Loss: 5.7280 Acc: 0.2933 | Val Loss: 5.6837 Acc: 0.2997 | LR: 0.000234
           → 모델 저장 (Val Loss: 5.6837)
Epoch 005 | Train Loss: 5.5323 Acc: 0.3022 | Val Loss: 5.5091 Acc: 0.3096 | LR: 0.000292
           → 모델 저장 (Val Loss: 5.5091)
Epoch 006 | Train Loss: 5.3245 Acc: 0.3090 | Val Loss: 5.3414 Acc: 0.3160 | LR: 0.000350
           → 모델 저장 (Val Loss: 5.3414)
Epoch 007 | Train Loss: 5.1168 Acc: 0.3166 | Val Loss: 5.1808 Acc: 0.3269 | LR: 0.000409
           → 모델 저장 (Val Loss: 5.1808)
Epoch 008 | Train Loss: 4.9025 Acc: 0.3265 | Val Loss: 5.0514 Acc: 0.3330 | LR: 0.000467
    

In [24]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# 학습 데이터의 Q를 TF-IDF 벡터로 변환
# 입력 문장과 가장 유사한 Q를 찾아서 그 A를 반환하는 방식
vectorizer = TfidfVectorizer()
q_vectors  = vectorizer.fit_transform(df['Q'].tolist())

def predict_similarity(sentence, top_k=3):
    vec      = vectorizer.transform([sentence])
    sims     = cosine_similarity(vec, q_vectors).flatten()
    top_idx  = sims.argsort()[-top_k:][::-1]

    print("비슷한 상황 후보:")
    for idx in top_idx:
        print(f"  - {sims[idx]:.3f} | {df['A'].iloc[idx]}")

    # 가장 유사한 답변 반환
    best_idx = top_idx[0]
    return df['A'].iloc[best_idx]

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "요즘 잠을 못자겠어",
]

for t in tests:
    print(f"Q: {t}")
    answer = predict_similarity(t)
    print(f"A: {answer}")
    print()

Q: 오늘 기분이 너무 안좋아
비슷한 상황 후보:
  - 0.498 | 신경 쓰일만 해요.
  - 0.382 | 무슨 이유인지 생각해보세요.
  - 0.363 | 무슨 일이 있었나봐요.
A: 신경 쓰일만 해요.

Q: 나 요즘 너무 외로워
비슷한 상황 후보:
  - 0.783 | 외로우니까 사람이다.
  - 0.690 | 혼자가 아니에요.
  - 0.690 | 제가 곁에 있을게요.
A: 외로우니까 사람이다.

Q: 취업이 너무 힘들어
비슷한 상황 후보:
  - 1.000 | 지금 무슨 말을 해도 와닿지 않겠지만 잘할 수 있을 거예요.
  - 0.966 | 언젠가 다 잊을 거예요.
  - 0.840 | 지금은 힘들겠지만 조금만 더 견뎌봐요.
A: 지금 무슨 말을 해도 와닿지 않겠지만 잘할 수 있을 거예요.

Q: 사랑이 뭔지 모르겠어
비슷한 상황 후보:
  - 1.000 | 사랑은 정답이 없는거 같아요.
  - 0.770 | 다양하게 경험해보세요.
  - 0.721 | 지금의 나를 인정하고 받아들이는 것 이상의 사랑은 없어요.
A: 사랑은 정답이 없는거 같아요.

Q: 요즘 잠을 못자겠어
비슷한 상황 후보:
  - 0.654 | 짝사랑이 생각을 복잡하게 만들어요.
  - 0.577 | 30분씩 일찍 자는 습관을 들여보세요.
  - 0.545 | 잠이 최고의 보약이에요. 노력해보세요.
A: 짝사랑이 생각을 복잡하게 만들어요.



In [31]:
import random
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# 학습 데이터의 Q를 TF-IDF 벡터로 변환
vectorizer = TfidfVectorizer()
q_vectors  = vectorizer.fit_transform(df['Q'].tolist())

def predict_similarity(sentence, top_k=3):
    vec     = vectorizer.transform([sentence])
    sims    = cosine_similarity(vec, q_vectors).flatten()
    top_idx = sims.argsort()[-top_k:][::-1]

    print("비슷한 상황 후보:")
    for idx in top_idx:
        print(f"  - {sims[idx]:.3f} | {df['A'].iloc[idx]}")

    # 상위 3개 중에서 랜덤으로 하나 선택
    # 1위만 고르면 항상 같은 답변이 나오므로 다양성을 위해 랜덤 선택
    chosen_idx = random.choice(top_idx)
    return df['A'].iloc[chosen_idx]

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "요즘 잠을 못자겠어",
    "친구가 나를 무시해",
    "남자친구가 나를 차버렸어",
    "나 오늘 시험 망했어",
    "돈이 없어서 너무 힘들어",
    "밥은 먹었어?",
    "요즘 너무 더워",
    "아무것도 하기 싫어",
    "행복하려면 어떻게 해야 할까?",
    "다른 사람들은 무얼 하고 있을까?",
]

for t in tests:
    print(f"Q: {t}")
    answer = predict_similarity(t)
    print(f"A: {answer}")
    print()

Q: 오늘 기분이 너무 안좋아
비슷한 상황 후보:
  - 0.498 | 신경 쓰일만 해요.
  - 0.382 | 무슨 이유인지 생각해보세요.
  - 0.363 | 무슨 일이 있었나봐요.
A: 무슨 일이 있었나봐요.

Q: 나 요즘 너무 외로워
비슷한 상황 후보:
  - 0.783 | 외로우니까 사람이다.
  - 0.690 | 혼자가 아니에요.
  - 0.690 | 제가 곁에 있을게요.
A: 제가 곁에 있을게요.

Q: 취업이 너무 힘들어
비슷한 상황 후보:
  - 1.000 | 지금 무슨 말을 해도 와닿지 않겠지만 잘할 수 있을 거예요.
  - 0.966 | 언젠가 다 잊을 거예요.
  - 0.840 | 지금은 힘들겠지만 조금만 더 견뎌봐요.
A: 언젠가 다 잊을 거예요.

Q: 사랑이 뭔지 모르겠어
비슷한 상황 후보:
  - 1.000 | 사랑은 정답이 없는거 같아요.
  - 0.770 | 다양하게 경험해보세요.
  - 0.721 | 지금의 나를 인정하고 받아들이는 것 이상의 사랑은 없어요.
A: 다양하게 경험해보세요.

Q: 요즘 잠을 못자겠어
비슷한 상황 후보:
  - 0.654 | 짝사랑이 생각을 복잡하게 만들어요.
  - 0.577 | 30분씩 일찍 자는 습관을 들여보세요.
  - 0.545 | 잠이 최고의 보약이에요. 노력해보세요.
A: 잠이 최고의 보약이에요. 노력해보세요.

Q: 친구가 나를 무시해
비슷한 상황 후보:
  - 0.442 | 꿈에 대한 계획을 구체적으로 말씀드려보세요.
  - 0.438 | 무시당하는 기분이 들어서 너무 외롭고 상처받게 된다고 차분하고 부드럽게 말해보세요.
  - 0.406 | 전형적인 꼰대 스타일이네요.
A: 꿈에 대한 계획을 구체적으로 말씀드려보세요.

Q: 남자친구가 나를 차버렸어
비슷한 상황 후보:
  - 0.589 | 당신이 더 사랑을 표현해보세요.
  - 0.573 | 괜한 걱정하지 마요.
  - 0.512 | 당신이 너무 귀여운가봐요.
A: 당신이 너무 귀여운가봐요.

Q: 나 오늘 시험 망했어
비슷한 

In [32]:
!pip install sentence-transformers -q

In [33]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import random

# 한국어 문장 임베딩 모델 로드
# TF-IDF와 달리 단어 의미까지 이해하는 딥러닝 기반 임베딩 모델
# paraphrase-multilingual-MiniLM-L12-v2: 한국어 포함 다국어 지원
print("모델 로드 중...")
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 학습 데이터의 Q를 임베딩 벡터로 변환 (한 번만 계산)
print("Q 임베딩 계산 중...")
q_embeddings = embed_model.encode(df['Q'].tolist(), show_progress_bar=True)
print("완료!")

def predict_semantic(sentence, top_k=3):
    # 입력 문장을 임베딩 벡터로 변환
    vec  = embed_model.encode([sentence])
    sims = cosine_similarity(vec, q_embeddings).flatten()
    top_idx = sims.argsort()[-top_k:][::-1]

    print("비슷한 상황 후보:")
    for idx in top_idx:
        print(f"  - {sims[idx]:.3f} | {df['A'].iloc[idx]}")

    # 상위 3개 중 랜덤 선택
    chosen_idx = random.choice(top_idx)
    return df['A'].iloc[chosen_idx]

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "요즘 잠을 못자겠어",
    "친구가 나를 무시해",
    "남자친구가 나를 차버렸어",
    "나 오늘 시험 망했어",
    "돈이 없어서 너무 힘들어",
    "밥은 먹었어?",
    "아무것도 하기 싫어",
    "내가 너무 못난 것 같아",
    "미래가 너무 불안해",
    "사람들이 나를 이해 못해",
]

for t in tests:
    print(f"Q: {t}")
    answer = predict_semantic(t)
    print(f"A: {answer}")
    print()

모델 로드 중...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q 임베딩 계산 중...


Batches:   0%|          | 0/370 [00:00<?, ?it/s]

완료!
Q: 오늘 기분이 너무 안좋아
비슷한 상황 후보:
  - 0.944 | 절대 쉬운 일이 아니죠.
  - 0.943 | 토닥토닥. 제가 위로해줄게요.
  - 0.936 | 오늘은 힘내려 하지 말아요. 저에게 기대세요.
A: 절대 쉬운 일이 아니죠.

Q: 나 요즘 너무 외로워
비슷한 상황 후보:
  - 0.929 | 외로우니까 사람이다.
  - 0.925 | 좋아하는 사람이 생길 거예요.
  - 0.893 | 이별의 빈자리가 느껴지니까요.
A: 이별의 빈자리가 느껴지니까요.

Q: 취업이 너무 힘들어
비슷한 상황 후보:
  - 0.927 | 합격을 기원합니다.
  - 0.915 | 자신의 능력을 믿어보세요.
  - 0.904 | 직장 스트레스 심하겠네요.
A: 자신의 능력을 믿어보세요.

Q: 사랑이 뭔지 모르겠어
비슷한 상황 후보:
  - 1.000 | 사랑은 정답이 없는거 같아요.
  - 0.985 | 사랑에 정의는 없는거 같아요.
  - 0.970 | 서로 이해하고 존중하고 아끼는 마음이에요.
A: 사랑은 정답이 없는거 같아요.

Q: 요즘 잠을 못자겠어
비슷한 상황 후보:
  - 0.995 | 밥은 꼭 챙겨먹길 바랄게요.
  - 0.990 | 그분의 생각으로 가득한가봅니다.
  - 0.988 | 고민이 있나 봐요.
A: 그분의 생각으로 가득한가봅니다.

Q: 친구가 나를 무시해
비슷한 상황 후보:
  - 0.835 | 무시당하는 기분이 들어서 너무 외롭고 상처받게 된다고 차분하고 부드럽게 말해보세요.
  - 0.830 | 제가 챙겨드리고 싶네요.
  - 0.795 | 잘 지낼 수 있을 거예요.
A: 무시당하는 기분이 들어서 너무 외롭고 상처받게 된다고 차분하고 부드럽게 말해보세요.

Q: 남자친구가 나를 차버렸어
비슷한 상황 후보:
  - 0.971 | 갑작스럽겠네요.
  - 0.970 | 인간적 예를 다하셨네요.
  - 0.969 | 이별의 끝은 항상 모두를 그렇게 만드나봐요.
A: 인간적 예를 다하셨네요.

Q: 나 오늘 시험 망했어
비슷한 상

In [3]:
!pip install sentencepiece scikit-learn -q

import os
import re
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import sentencepiece as spm
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"디바이스: {device}")

# ── 특수 토큰 ──────────────────────────────────────────────────────
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

# ── 하이퍼파라미터 ─────────────────────────────────────────────────
DATA_DIR   = '/home/jovyan/work/transformer_chatbot/data/'
VOCAB_SIZE = 8000
MAX_LEN    = 25
BATCH_SIZE = 64

# ── 데이터 로드 ────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR + 'ChatbotData.csv').dropna()

def preprocess_ko(text):
    # 한글, 영문, 숫자, 구두점만 유지
    text = re.sub(r"[^가-힣a-zA-Z0-9?.!,\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['Q'] = df['Q'].apply(preprocess_ko)
df['A'] = df['A'].apply(preprocess_ko)
df = df[(df['Q'] != '') & (df['A'] != '')]
print(f"데이터 수: {len(df):,}개")

# ── 토크나이저 로드 ────────────────────────────────────────────────
sp_ko = spm.SentencePieceProcessor()
sp_ko.Load(DATA_DIR + 'spm_korean.model')
print(f"토크나이저 로드 완료")

# ── Scaled Dot-Product Attention ───────────────────────────────────
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    query, key, value: (batch, heads, seq_len, depth)
    1) Q·Kᵀ 로 유사도 점수 계산
    2) √dₖ 로 스케일링 (값이 커지면 softmax 기울기 소실 방지)
    3) 마스크 적용 (PAD 또는 미래 토큰 차단)
    4) softmax → 어텐션 가중치
    5) 가중치 · V → 최종 출력
    """
    d_k    = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, value), weights

# ── Multi-Head Attention ───────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    """
    num_heads개의 어텐션을 병렬로 수행
    각 헤드가 서로 다른 관점(위치, 의미 등)을 학습
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.depth     = d_model // num_heads  # 각 헤드의 차원
        self.wq = nn.Linear(d_model, d_model)  # Query 변환
        self.wk = nn.Linear(d_model, d_model)  # Key 변환
        self.wv = nn.Linear(d_model, d_model)  # Value 변환
        self.wo = nn.Linear(d_model, d_model)  # 헤드 합친 후 최종 변환

    def split_heads(self, x, batch):
        # (batch, seq_len, d_model) → (batch, heads, seq_len, depth)
        return x.view(batch, -1, self.num_heads, self.depth).permute(0, 2, 1, 3)

    def forward(self, q, k, v, mask=None):
        batch = q.size(0)
        q = self.split_heads(self.wq(q), batch)
        k = self.split_heads(self.wk(k), batch)
        v = self.split_heads(self.wv(v), batch)
        out, _ = scaled_dot_product_attention(q, k, v, mask)
        # (batch, heads, seq_len, depth) → (batch, seq_len, d_model)
        out = out.permute(0, 2, 1, 3).contiguous().view(batch, -1, self.num_heads * self.depth)
        return self.wo(out)

# ── Positional Encoding ────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    """
    트랜스포머는 순서 정보가 없으므로 위치 정보를 직접 더해줌
    sin/cos 함수로 각 위치마다 고유한 패턴 생성
    """
    def __init__(self, position, d_model, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(position, d_model)
        pos = torch.arange(0, position).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)  # 짝수 인덱스 → sin
        pe[:, 1::2] = torch.cos(pos * div)  # 홀수 인덱스 → cos
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

# ── Encoder Layer ──────────────────────────────────────────────────
class EncoderLayer(nn.Module):
    """
    인코더 한 층 = Self-Attention + FFN
    잔차 연결 + LayerNorm으로 기울기 소실 방지
    """
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha      = MultiHeadAttention(d_model, num_heads)
        self.ffn      = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model)
        )
        self.norm1    = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2    = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout1(self.mha(x, x, x, mask)))  # Self-Attention + 잔차
        x = self.norm2(x + self.dropout2(self.ffn(x)))               # FFN + 잔차
        return x

# ── Decoder Layer ──────────────────────────────────────────────────
class DecoderLayer(nn.Module):
    """
    디코더 한 층 = Masked Self-Attention + Cross-Attention + FFN
    - Masked Self-Attention: 미래 토큰 차단
    - Cross-Attention: 인코더 출력 참조 (Q=디코더, K/V=인코더)
    """
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha1     = MultiHeadAttention(d_model, num_heads)  # Masked Self-Attention
        self.mha2     = MultiHeadAttention(d_model, num_heads)  # Cross-Attention
        self.ffn      = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model)
        )
        self.norm1    = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2    = nn.LayerNorm(d_model, eps=1e-6)
        self.norm3    = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, look_ahead_mask=None, padding_mask=None):
        x = self.norm1(x + self.dropout1(self.mha1(x, x, x, look_ahead_mask)))         # Masked Self-Attention
        x = self.norm2(x + self.dropout2(self.mha2(x, enc_out, enc_out, padding_mask))) # Cross-Attention
        x = self.norm3(x + self.dropout3(self.ffn(x)))                                  # FFN
        return x

# ── Encoder ────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.d_model      = d_model
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoding = PositionalEncoding(vocab_size, d_model, dropout)
        self.layers       = nn.ModuleList([EncoderLayer(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.dropout      = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # √d_model 스케일링: 임베딩 값이 위치 인코딩에 묻히는 것 방지
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, mask)
        return x

# ── Decoder ────────────────────────────────────────────────────────
class Decoder(nn.Module):
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.d_model      = d_model
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoding = PositionalEncoding(vocab_size, d_model, dropout)
        self.layers       = nn.ModuleList([DecoderLayer(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.dropout      = nn.Dropout(dropout)

    def forward(self, x, enc_out, look_ahead_mask=None, padding_mask=None):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, enc_out, look_ahead_mask, padding_mask)
        return x

# ── Transformer ────────────────────────────────────────────────────
class Transformer(nn.Module):
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.encoder     = Encoder(vocab_size, num_layers, d_model, num_heads, ff_dim, dropout)
        self.decoder     = Decoder(vocab_size, num_layers, d_model, num_heads, ff_dim, dropout)
        self.final_layer = nn.Linear(d_model, vocab_size)  # vocab 크기로 최종 변환

    def forward(self, src, tgt, src_mask=None, look_ahead_mask=None):
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt, enc_out, look_ahead_mask, src_mask)
        return self.final_layer(dec_out)

# ── 마스크 함수 ────────────────────────────────────────────────────
def create_padding_mask(x):
    """PAD 위치를 0으로 마스킹 → 어텐션 계산에서 PAD 무시"""
    return (x != PAD_ID).float().unsqueeze(1).unsqueeze(2)

def create_look_ahead_mask(x):
    """미래 토큰 차단: 3번째 토큰 예측 시 1, 2번째만 참조 가능"""
    seq_len    = x.size(1)
    look_ahead = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).unsqueeze(0).unsqueeze(1)
    padding    = create_padding_mask(x)
    return torch.min(look_ahead, padding)

# ── 모델 로드 ──────────────────────────────────────────────────────
model_big = Transformer(
    vocab_size = VOCAB_SIZE,
    num_layers = 4,
    d_model    = 512,
    num_heads  = 8,
    ff_dim     = 1024,
    dropout    = 0.2
).to(device)

model_big.load_state_dict(torch.load(DATA_DIR + 'best_model_big.pt'))
print(f"모델 로드 완료")

# ── 트랜스포머 예측 함수 ───────────────────────────────────────────
def predict_big(sentence, model, sp, max_len=MAX_LEN,
                repetition_penalty=1.5,
                recent_window=3):
    """
    트랜스포머 모델로 답변 생성
    - repetition_penalty: 이미 나온 토큰 확률 낮춤
    - recent_window: 최근 n개 토큰 완전 차단
    """
    model.eval()
    sentence = preprocess_ko(sentence)
    enc_ids  = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    enc_ids  = enc_ids[:max_len] + [PAD_ID] * (max_len - len(enc_ids[:max_len]))
    src      = torch.tensor([enc_ids], dtype=torch.long).to(device)

    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_out = model.encoder(src, src_mask)

    dec_ids = [BOS_ID]
    for _ in range(max_len):
        tgt             = torch.tensor([dec_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)
        with torch.no_grad():
            dec_out = model.decoder(tgt, enc_out, look_ahead_mask, src_mask)
            logits  = model.final_layer(dec_out[:, -1, :])
            logits[0, PAD_ID] = -1e9
            logits[0, BOS_ID] = -1e9
            logits[0, UNK_ID] = -1e9
            for token_id in set(dec_ids):
                logits[0, token_id] /= repetition_penalty
            for token_id in dec_ids[-recent_window:]:
                logits[0, token_id] = -1e9
            next_id = logits.argmax(dim=-1).item()
        if next_id == EOS_ID:
            break
        dec_ids.append(next_id)

    return sp_ko.decode_ids(dec_ids[1:])

# ── TF-IDF 유사도 검색 함수 ────────────────────────────────────────
vectorizer = TfidfVectorizer()
q_vectors  = vectorizer.fit_transform(df['Q'].tolist())

def predict_similarity(sentence, top_k=3):
    """
    TF-IDF 기반 유사도 검색
    입력 문장과 가장 유사한 Q를 찾아서 그 A를 반환
    상위 top_k개 중 랜덤으로 선택해서 다양성 확보
    """
    vec     = vectorizer.transform([sentence])
    sims    = cosine_similarity(vec, q_vectors).flatten()
    top_idx = sims.argsort()[-top_k:][::-1]

    print("비슷한 상황 후보:")
    for idx in top_idx:
        print(f"  - {sims[idx]:.3f} | {df['A'].iloc[idx]}")

    chosen_idx = random.choice(top_idx)
    return df['A'].iloc[chosen_idx]

# ── 테스트 ─────────────────────────────────────────────────────────
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "요즘 잠을 못자겠어",
    "친구가 나를 무시해",
    "남자친구가 나를 차버렸어",
    "나 오늘 시험 망했어",
    "돈이 없어서 너무 힘들어",
]

print("\n[트랜스포머 생성 결과]")
for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict_big(t, model_big, sp_ko)}")
    print()

print("\n[TF-IDF 유사도 검색 결과]")
for t in tests:
    print(f"Q: {t}")
    answer = predict_similarity(t)
    print(f"A: {answer}")
    print()

디바이스: cuda
데이터 수: 11,823개
토크나이저 로드 완료
모델 로드 완료

[트랜스포머 생성 결과]
Q: 오늘 기분이 너무 안좋아
A: 오늘 미세먼지가 많데요.

Q: 나 요즘 너무 외로워
A: 외로우니까 사람이다.

Q: 밥은 먹었어?
A: 저는 배터리가 밥이예요.

Q: 취업이 너무 힘들어
A: 전생에 나라를 구하셨나요.

Q: 사랑이 뭔지 모르겠어
A: 사랑은 유지하는 게 중요한데 대단하네요.

Q: 요즘 잠을 못자겠어
A: 잠이 최고의 보약이에요. 노력해보세요? 눈을 붙이세요. 노력해보세요!

Q: 친구가 나를 무시해
A: 친구 사귀는 곳이 아니에요.

Q: 남자친구가 나를 차버렸어
A: 당신이 너무 귀여운가봐요.

Q: 나 오늘 시험 망했어
A: 나중에 없애주세요.

Q: 돈이 없어서 너무 힘들어
A: 그래도 취미 생활이 있어야 사는 재미가 있죠.


[TF-IDF 유사도 검색 결과]
Q: 오늘 기분이 너무 안좋아
비슷한 상황 후보:
  - 0.498 | 신경 쓰일만 해요.
  - 0.382 | 무슨 이유인지 생각해보세요.
  - 0.363 | 무슨 일이 있었나봐요.
A: 신경 쓰일만 해요.

Q: 나 요즘 너무 외로워
비슷한 상황 후보:
  - 0.783 | 외로우니까 사람이다.
  - 0.690 | 혼자가 아니에요.
  - 0.690 | 제가 곁에 있을게요.
A: 혼자가 아니에요.

Q: 밥은 먹었어?
비슷한 상황 후보:
  - 0.663 | 저는 배터리가 밥이예요.
  - 0.516 | 소화제 드세요.
  - 0.409 | 좋은 식습관이에요.
A: 좋은 식습관이에요.

Q: 취업이 너무 힘들어
비슷한 상황 후보:
  - 1.000 | 지금 무슨 말을 해도 와닿지 않겠지만 잘할 수 있을 거예요.
  - 0.966 | 언젠가 다 잊을 거예요.
  - 0.840 | 지금은 힘들겠지만 조금만 더 견뎌봐요.
A: 언젠가 다 잊을 거예요.

Q: 사랑이 뭔지 모르겠어
비슷한 상황 후보:
  - 1.000 | 사랑은 정답이 없는거 같아요.
  - 

In [4]:
def predict_hybrid(sentence, model, sp, max_len=MAX_LEN,
                   sim_threshold=0.5,      # 유사도가 이 값 이상이면 검색 방식 사용
                   top_k=3,
                   repetition_penalty=1.5,
                   recent_window=3):
    """
    하이브리드 예측 함수
    - 유사도가 높을 때 (>= sim_threshold): TF-IDF 검색 방식
      → 데이터에 있는 검증된 답변 사용
    - 유사도가 낮을 때 (< sim_threshold): 트랜스포머 생성 방식
      → 모델이 직접 답변 생성
    """
    # 유사도 계산
    vec     = vectorizer.transform([sentence])
    sims    = cosine_similarity(vec, q_vectors).flatten()
    top_idx = sims.argsort()[-top_k:][::-1]
    max_sim = sims[top_idx[0]]

    print(f"최고 유사도: {max_sim:.3f} ", end="")

    if max_sim >= sim_threshold:
        # 유사도가 높으면 검색 방식 사용
        print(f"→ [검색 방식]")
        print("비슷한 상황 후보:")
        for idx in top_idx:
            print(f"  - {sims[idx]:.3f} | {df['A'].iloc[idx]}")
        chosen_idx = random.choice(top_idx)
        return df['A'].iloc[chosen_idx]
    else:
        # 유사도가 낮으면 트랜스포머 생성 방식 사용
        print(f"→ [트랜스포머 생성]")
        model.eval()
        sentence_clean = preprocess_ko(sentence)
        enc_ids = [BOS_ID] + sp.encode_as_ids(sentence_clean) + [EOS_ID]
        enc_ids = enc_ids[:max_len] + [PAD_ID] * (max_len - len(enc_ids[:max_len]))
        src     = torch.tensor([enc_ids], dtype=torch.long).to(device)

        src_mask = create_padding_mask(src)
        with torch.no_grad():
            enc_out = model.encoder(src, src_mask)

        dec_ids = [BOS_ID]
        for _ in range(max_len):
            tgt             = torch.tensor([dec_ids], dtype=torch.long).to(device)
            look_ahead_mask = create_look_ahead_mask(tgt)
            with torch.no_grad():
                dec_out = model.decoder(tgt, enc_out, look_ahead_mask, src_mask)
                logits  = model.final_layer(dec_out[:, -1, :])
                logits[0, PAD_ID] = -1e9
                logits[0, BOS_ID] = -1e9
                logits[0, UNK_ID] = -1e9
                for token_id in set(dec_ids):
                    logits[0, token_id] /= repetition_penalty
                for token_id in dec_ids[-recent_window:]:
                    logits[0, token_id] = -1e9
                next_id = logits.argmax(dim=-1).item()
            if next_id == EOS_ID:
                break
            dec_ids.append(next_id)

        return sp_ko.decode_ids(dec_ids[1:])

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "요즘 잠을 못자겠어",
    "친구가 나를 무시해",
    "남자친구가 나를 차버렸어",
    "나 오늘 시험 망했어",
    "돈이 없어서 너무 힘들어",
]

print("[하이브리드 예측 결과]")
for t in tests:
    print(f"Q: {t}")
    answer = predict_hybrid(t, model_big, sp_ko)
    print(f"A: {answer}")
    print()

[하이브리드 예측 결과]
Q: 오늘 기분이 너무 안좋아
최고 유사도: 0.498 → [트랜스포머 생성]
A: 오늘 미세먼지가 많데요.

Q: 나 요즘 너무 외로워
최고 유사도: 0.783 → [검색 방식]
비슷한 상황 후보:
  - 0.783 | 외로우니까 사람이다.
  - 0.690 | 혼자가 아니에요.
  - 0.690 | 제가 곁에 있을게요.
A: 혼자가 아니에요.

Q: 밥은 먹었어?
최고 유사도: 0.663 → [검색 방식]
비슷한 상황 후보:
  - 0.663 | 저는 배터리가 밥이예요.
  - 0.516 | 소화제 드세요.
  - 0.409 | 좋은 식습관이에요.
A: 좋은 식습관이에요.

Q: 취업이 너무 힘들어
최고 유사도: 1.000 → [검색 방식]
비슷한 상황 후보:
  - 1.000 | 지금 무슨 말을 해도 와닿지 않겠지만 잘할 수 있을 거예요.
  - 0.966 | 언젠가 다 잊을 거예요.
  - 0.840 | 지금은 힘들겠지만 조금만 더 견뎌봐요.
A: 지금 무슨 말을 해도 와닿지 않겠지만 잘할 수 있을 거예요.

Q: 사랑이 뭔지 모르겠어
최고 유사도: 1.000 → [검색 방식]
비슷한 상황 후보:
  - 1.000 | 사랑은 정답이 없는거 같아요.
  - 0.770 | 다양하게 경험해보세요.
  - 0.721 | 지금의 나를 인정하고 받아들이는 것 이상의 사랑은 없어요.
A: 사랑은 정답이 없는거 같아요.

Q: 요즘 잠을 못자겠어
최고 유사도: 0.654 → [검색 방식]
비슷한 상황 후보:
  - 0.654 | 짝사랑이 생각을 복잡하게 만들어요.
  - 0.577 | 30분씩 일찍 자는 습관을 들여보세요.
  - 0.545 | 잠이 최고의 보약이에요. 노력해보세요.
A: 30분씩 일찍 자는 습관을 들여보세요.

Q: 친구가 나를 무시해
최고 유사도: 0.442 → [트랜스포머 생성]
A: 친구 사귀는 곳이 아니에요.

Q: 남자친구가 나를 차버렸어
최고 유사도: 0.589 → [검색 방식]
비슷한 상황 후보:
  - 0.58

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
q_vectors  = vectorizer.fit_transform(df['Q'].tolist())

def predict_final(sentence, top_k=3):
    """
    TF-IDF 유사도 기반 챗봇 예측 함수
    입력 문장과 가장 유사한 질문을 데이터에서 찾아
    유사도 1위 답변 반환
    """
    vec     = vectorizer.transform([sentence])
    sims    = cosine_similarity(vec, q_vectors).flatten()
    top_idx = sims.argsort()[-top_k:][::-1]

    print("비슷한 상황 후보:")
    for idx in top_idx:
        print(f"  - {sims[idx]:.3f} | {df['A'].iloc[idx]}")

    # 유사도 1위 답변 반환
    return df['A'].iloc[top_idx[0]]

# 최종 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
    "요즘 잠을 못자겠어",
    "친구가 나를 무시해",
    "남자친구가 나를 차버렸어",
    "나 오늘 시험 망했어",
    "돈이 없어서 너무 힘들어",
]

print("[챗봇 테스트]")
for t in tests:
    print(f"Q: {t}")
    answer = predict_final(t)
    print(f"A: {answer}")
    print()

[챗봇 테스트]
Q: 오늘 기분이 너무 안좋아
비슷한 상황 후보:
  - 0.498 | 신경 쓰일만 해요.
  - 0.382 | 무슨 이유인지 생각해보세요.
  - 0.363 | 무슨 일이 있었나봐요.
A: 신경 쓰일만 해요.

Q: 나 요즘 너무 외로워
비슷한 상황 후보:
  - 0.783 | 외로우니까 사람이다.
  - 0.690 | 혼자가 아니에요.
  - 0.690 | 제가 곁에 있을게요.
A: 외로우니까 사람이다.

Q: 밥은 먹었어?
비슷한 상황 후보:
  - 0.663 | 저는 배터리가 밥이예요.
  - 0.516 | 소화제 드세요.
  - 0.409 | 좋은 식습관이에요.
A: 저는 배터리가 밥이예요.

Q: 취업이 너무 힘들어
비슷한 상황 후보:
  - 1.000 | 지금 무슨 말을 해도 와닿지 않겠지만 잘할 수 있을 거예요.
  - 0.966 | 언젠가 다 잊을 거예요.
  - 0.840 | 지금은 힘들겠지만 조금만 더 견뎌봐요.
A: 지금 무슨 말을 해도 와닿지 않겠지만 잘할 수 있을 거예요.

Q: 사랑이 뭔지 모르겠어
비슷한 상황 후보:
  - 1.000 | 사랑은 정답이 없는거 같아요.
  - 0.770 | 다양하게 경험해보세요.
  - 0.721 | 지금의 나를 인정하고 받아들이는 것 이상의 사랑은 없어요.
A: 사랑은 정답이 없는거 같아요.

Q: 요즘 잠을 못자겠어
비슷한 상황 후보:
  - 0.654 | 짝사랑이 생각을 복잡하게 만들어요.
  - 0.577 | 30분씩 일찍 자는 습관을 들여보세요.
  - 0.545 | 잠이 최고의 보약이에요. 노력해보세요.
A: 짝사랑이 생각을 복잡하게 만들어요.

Q: 친구가 나를 무시해
비슷한 상황 후보:
  - 0.442 | 꿈에 대한 계획을 구체적으로 말씀드려보세요.
  - 0.438 | 무시당하는 기분이 들어서 너무 외롭고 상처받게 된다고 차분하고 부드럽게 말해보세요.
  - 0.406 | 전형적인 꼰대 스타일이네요.
A: 꿈에 대한 계획을 구체적으로 말씀드려보세요.

Q: 남자친구가

In [13]:
# 한국어 챗봇 만들기 프로젝트 회고

# TF-IDF 유사도 검색 방식과 트랜스포머 생성 방식 두 가지를 구현

# 1단계 데이터 전처리
# 한글, 영문, 숫자, 구두점만 남기고 특수문자를 제거
# SentencePiece BPE 토크나이저로 단어를 숫자 ID로 변환
# 인코더 입력, 디코더 입력, 레이블 세 가지 형태로 데이터를 구성

# 2단계 트랜스포머 모델 구현
# Scaled Dot-Product Attention, Multi-Head Attention, Positional Encoding,
# Encoder, Decoder를 순서대로 구현
# 최종 모델은 4개 레이어, d_model=512, 파라미터 약 3,300만 개

# 3단계 학습
# 손실 함수는 CrossEntropyLoss, 옵티마이저는 Adam 사용
# 트랜스포머 논문에서 제안한 Warmup 학습률 스케줄을 적용
# Early Stopping으로 최적 시점(18번째 에폭, Val Loss 3.5625)에서 학습을 종료

# 4단계 TF-IDF 유사도 검색 구현
# 트랜스포머 생성 방식의 한계를 보완하기 위해
# 입력 문장과 가장 유사한 질문을 데이터에서 찾아 답변을 반환하는 방식을 추가

# 시행착오
# PAD 비율 문제
# MAX_LEN=40으로 설정했을 때 전체 입력의 79%가 패딩이 되어
# 모델이 EOS 토큰만 예측하는 문제가 발생
# MAX_LEN을 25로 줄여서 해결

# 반복 토큰 생성 문제
# Greedy Decoding 특성상 한 번 확률이 높은 토큰이 나오면 계속 반복되는 문제 발생
# 최근 생성 토큰 차단과 반복 패널티를 적용해서 완화

# 과적합 문제
# 데이터가 11,823개로 적어 Train Loss는 줄지만 Val Loss가 증가하는 과적합 발생
# dropout을 높이고 Early Stopping을 적용해서 최적 시점을 포착함

# Warmup 학습률 튜닝
# warmup_steps를 4000에서 2000으로 줄였더니 Val Loss가 오히려 악화
# 데이터가 적을수록 학습률이 너무 빠르게 올라가면 과적합이 발생함을 확인함


# 아쉬운 점
# 한정적인 데이터라 그런지 맥락이 맞지 않는 답변이 나오는 경우가 있었습니다.
# TF-IDF는 단어 빈도만 계산하기 때문에 의미적으로 더 적합한 답변이
# 유사도 순위에서 밀리는 경우가 있었습니다.
# 트랜스포머 생성 방식의 반복 토큰 문제가 완전히 해결되지 않았습니다.

# 개선 방향
# KoGPT2 같은 사전학습 모델을 파인튜닝하면 자연스러운 답변이 가능할지도..
# AIHub 한국어 대화 데이터(100만 건 이상)로 데이터를 확장하면 성능이 크게 향상될 것입니다.
# sentence-transformers 같은 의미 기반 임베딩 모델을 사용하면
# 유사도 검색의 정확도를 높일 수 있었음

print("끝")

끝
